In [25]:
from pathlib import Path
from typing import List, Tuple, Dict
import json, random, itertools, os, gc, statistics as st
import seqeval

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import KFold, train_test_split
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import BallTree
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity, pairwise_distances
from sklearn.cluster import KMeans

import torch
from transformers import (
    AutoTokenizer,
        AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

from seqeval.metrics import f1_score, classification_report
from evaluate import load as load_metric
from tqdm.auto import tqdm

# Configuração e Verificação Inicial

In [2]:
SEED_GLOBAL = 42
random.seed(SEED_GLOBAL)
np.random.seed(SEED_GLOBAL)
torch.manual_seed(SEED_GLOBAL)

CSV_PATH = Path("data/cachacaNER.csv")  # ajuste se necessário
MODEL_NAME = "neuralmind/bert-base-portuguese-cased"

df = pd.read_csv(CSV_PATH)

In [3]:
for cand in ["sentence_id", "sentence", "sent_id"]:
    if cand in df.columns:
        SENT_COL = cand
        break
else:
    raise KeyError(f"Coluna de sentença não encontrada em {list(df.columns)}")


def sent_to_record(sent_id, g):
    return {
        "sentence_id": int(sent_id),
        "tokens": g["token"].tolist(),
        "ner_tags": g["tag"].tolist(),
    }


In [4]:
records = [sent_to_record(i, g) for i, g in df.groupby(SENT_COL, sort=False)]
cachaca_full = Dataset.from_list(records)

In [5]:
cachaca_full

Dataset({
    features: ['sentence_id', 'tokens', 'ner_tags'],
    num_rows: 13628
})

In [6]:
# lista de rótulos (ordem alfabética garante consistência entre runs)
label_list = sorted({lab for sent in cachaca_full["ner_tags"] for lab in sent})
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
NUM_LABELS = len(label_list)

In [7]:
id2label

{0: 'B-CARACTERISTICA_SENSORIAL_AROMA',
 1: 'B-CARACTERISTICA_SENSORIAL_CONSISTÊNCIA',
 2: 'B-CARACTERISTICA_SENSORIAL_COR',
 3: 'B-CARACTERISTICA_SENSORIAL_SABOR',
 4: 'B-CLASSIFICACAO_BEBIDA',
 5: 'B-EQUIPAMENTO_DESTILACAO',
 6: 'B-GRADUACAO_ALCOOLICA',
 7: 'B-NOME_BEBIDA',
 8: 'B-NOME_LOCAL',
 9: 'B-NOME_ORGANIZACAO',
 10: 'B-NOME_PESSOA',
 11: 'B-PRECO',
 12: 'B-RECIPIENTE_ARMAZENAMENTO',
 13: 'B-TEMPO',
 14: 'B-TEMPO_ARMAZENAMENTO',
 15: 'B-TIPO_MADEIRA',
 16: 'B-VOLUME',
 17: 'I-CARACTERISTICA_SENSORIAL_AROMA',
 18: 'I-CARACTERISTICA_SENSORIAL_CONSISTÊNCIA',
 19: 'I-CARACTERISTICA_SENSORIAL_COR',
 20: 'I-CARACTERISTICA_SENSORIAL_SABOR',
 21: 'I-CLASSIFICACAO_BEBIDA',
 22: 'I-EQUIPAMENTO_DESTILACAO',
 23: 'I-GRADUACAO_ALCOOLICA',
 24: 'I-NOME_BEBIDA',
 25: 'I-NOME_LOCAL',
 26: 'I-NOME_ORGANIZACAO',
 27: 'I-NOME_PESSOA',
 28: 'I-PRECO',
 29: 'I-RECIPIENTE_ARMAZENAMENTO',
 30: 'I-TEMPO',
 31: 'I-TEMPO_ARMAZENAMENTO',
 32: 'I-TIPO_MADEIRA',
 33: 'I-VOLUME',
 34: 'O'}

In [8]:
NUM_LABELS

35

# Splits

In [9]:
def split_standard(ds: Dataset) -> DatasetDict:
    """Usa coluna trainingTest do CSV (80/20 original)."""
    if "trainingTest" not in df.columns:
        raise ValueError("CSV não contém a coluna 'trainingTest'")
    train_ids = df.loc[df["trainingTest"] == "training", SENT_COL].unique()
    test_ids = df.loc[df["trainingTest"] == "test", SENT_COL].unique()
    return DatasetDict(
        train=ds.filter(lambda ex: ex["sentence_id"] in train_ids),
        dev=ds.filter(lambda ex: ex["sentence_id"] in test_ids),
    )

In [10]:
def random_splits(
    ds: Dataset, test_size=0.2, seeds: List[int] = range(30)
) -> List[DatasetDict]:
    triples = []
    for s in seeds:
        train, dev = ds.train_test_split(test_size=test_size, seed=s).values()
        triples.append(DatasetDict(train=train, dev=dev))
    return triples

In [11]:
# def split_heur_length(ds: Dataset, top_pct: float = 0.20) -> DatasetDict:
#     lengths = np.array([len(t) for t in ds["tokens"]])
#     thr = np.percentile(lengths, 100 * (1 - top_pct))
#     mask = lengths >= thr
#     return DatasetDict(train=ds.filter(~mask), dev=ds.filter(mask))


def split_heur_length(ds: Dataset, top_pct: float = 0.20) -> DatasetDict:
    """20 % das sentenças mais longas viram conjunto de validação (dev)."""
    lengths = np.array([len(t) for t in ds["tokens"]])
    thr = np.percentile(lengths, 100 * (1 - top_pct))  
    mask = lengths >= thr  

    dev_idx = np.where(mask)[0].tolist()  # índices → list[int]
    train_idx = np.where(~mask)[0].tolist()

    return DatasetDict(
        train=ds.select(train_idx),
        dev=ds.select(dev_idx),
    )

    # Tamanho da sentenças


# def split_heur_rare(ds: Dataset, freq_thr: int = 5) -> DatasetDict:
#     freq = Counter(w.lower() for sent in ds["tokens"] for w in sent)
#     rare = {w for w, c in freq.items() if c <= freq_thr}

#     def has_rare(example):
#         return any(w.lower() in rare for w in example["tokens"])

#     return DatasetDict(
#         train=ds.filter(lambda ex: not has_rare(ex)), dev=ds.filter(has_rare)
#     )


def split_heur_rare(ds: Dataset, freq_thr: int = 5) -> DatasetDict:
    freq = Counter(w.lower() for sent in ds["tokens"] for w in sent)
    rare = {w for w, c in freq.items() if c <= freq_thr}

    keep_dev = []
    for sent in ds["tokens"]:
        print(sent)
        keep_dev.append(any(w.lower() in rare for w in sent))

    dev_idx = [i for i, x in enumerate(keep_dev) if x]
    train_idx = [i for i, x in enumerate(keep_dev) if not x]

    return DatasetDict(
        train=ds.select(train_idx),
        dev=ds.select(dev_idx),
    )

    # Raridade dos tokens

In [12]:
def split_adversarial(ds: Dataset, pct_test: float = 0.20) -> DatasetDict:
    k = int(len(ds) * pct_test)
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    embeds = model.encode([" ".join(t) for t in ds["tokens"]], show_progress_bar=False)
    tree = BallTree(embeds, leaf_size=40)

    idx_train, idx_test = set(range(len(ds))), []
    # semente = ponto mais central
    seed_idx = np.argmax(np.linalg.norm(embeds - embeds.mean(0), axis=1))
    idx_train.remove(seed_idx)
    idx_test.append(seed_idx)
    print(k)
    while len(idx_test) < k:
        print(len(idx_test))
        dists, _ = tree.query(embeds[list(idx_train)], k=1, return_distance=True)
        nxt = list(idx_train)[int(np.argmax(dists))]
        idx_train.remove(nxt)
        idx_test.append(nxt)

    return DatasetDict(
        train=ds.select(sorted(idx_train)),
        dev=ds.select(sorted(idx_test)),
    )

    # Maximizando Wassertein Distance


def split_adversarial_fast(ds: Dataset, pct_test: float = 0.20) -> DatasetDict:
    """
    Farthest-Point Sampling aproximando Wasserstein – versão vetorizada.
    Seleciona pct_test (~20 %) das sentenças como conjunto 'dev'.
    """
    k = int(len(ds) * pct_test)
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    embeds = model.encode(
        [" ".join(t) for t in ds["tokens"]],
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,  # acelera distância euclidiana ≈ cos
    )

    n = embeds.shape[0]
    idx_all = np.arange(n)

    # 1) ponto mais "central" (norma mais distante da média)
    seed_idx = np.argmax(np.linalg.norm(embeds - embeds.mean(0), axis=1))
    selected = [seed_idx]

    # 2) vetor de distâncias mínimas a qualquer ponto já escolhido
    min_dists = np.linalg.norm(embeds - embeds[seed_idx], axis=1)

    while len(selected) < k:
        next_idx = np.argmax(min_dists)
        selected.append(next_idx)

        # atualiza min_dists com a distância ao novo ponto — tudo de uma vez
        d_new = np.linalg.norm(embeds - embeds[next_idx], axis=1)
        min_dists = np.minimum(min_dists, d_new)

    train_idx = np.setdiff1d(idx_all, selected, assume_unique=True)

    return DatasetDict(
        train=ds.select(train_idx.tolist()),
        dev=ds.select(selected),
    )

In [13]:
# def loc_split(
#     dataset: Dataset, pct_test: float = 0.20, ngram: int = 4, seed: int = 42
# ) -> DatasetDict:
#     """
#     Split baseado em baixa sobreposição léxica (4-gram Jaccard).
#     Teste = pct_test das sentenças com menor overlap em relação ao pool.
#     """
#     # 1. Texto plano por sentença
#     docs = [" ".join(toks) for toks in dataset["tokens"]]

#     # 2. Vetorizar 4-grams (binário)
#     vect = CountVectorizer(
#         analyzer="word", ngram_range=(ngram, ngram), binary=True
#     ).fit(docs)
#     X = vect.transform(docs)

#     # 3. Similaridade Jaccard aproximada com matriz binária
#     # Jaccard(A,B) = |A∩B|/|A∪B| = 1 - |AΔB|/|A∪B|
#     # Usamos: overlap = (A·Bᵀ) / (|A|+|B|-A·Bᵀ)
#     bin_counts = X.sum(axis=1).A1

#     # Para cada doc i, escolhemos vizinho + próximo (fast):
#     from sklearn.metrics.pairwise import cosine_similarity

#     # (cosine no binário ∝ |A∩B|)
#     sim = cosine_similarity(X, dense_output=False)
#     # Soma dos top-k overlaps (k=5) como score
#     k = 5
#     topk = np.zeros(len(dataset))
#     for i in range(sim.shape[0]):
#         row = sim.getrow(i).toarray()[0]
#         idx = np.argpartition(-row, range(1, k + 1))[1 : k + 1]
#         # overlap ≈ |∩|
#         inter = row[idx] * bin_counts[i]
#         uni = bin_counts[i] + bin_counts[idx] - inter
#         topk[i] = (inter / uni).mean()

#     # 4. Ordenar por overlap crescente ⇒ mais “novos” vão p/ teste
#     order = np.argsort(topk)
#     n_test = int(len(dataset) * pct_test)
#     test_idx = order[:n_test]
#     train_idx = order[n_test:]

#     return DatasetDict(
#         {"train": dataset.select(train_idx), "test": dataset.select(test_idx)}
#     )

In [14]:
def loc_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    ngram: int = 4,
    seed: int = 42,
) -> DatasetDict:
    """
    Divide por sobreposição léxica (n-gram Jaccard).
    Frações independentes para teste e validação.
    """
    docs = [" ".join(toks) for toks in dataset["tokens"]]

    vect = CountVectorizer(
        analyzer="word", ngram_range=(ngram, ngram), binary=True
    ).fit(docs)
    X = vect.transform(docs)
    bin_counts = X.sum(axis=1).A1

    sim = cosine_similarity(X, dense_output=False)
    k = 5
    topk = np.zeros(len(dataset))
    for i in range(sim.shape[0]):
        row = sim.getrow(i).toarray()[0]
        idx = np.argpartition(-row, range(1, k + 1))[1 : k + 1]
        inter = row[idx] * bin_counts[i]
        uni = bin_counts[i] + bin_counts[idx] - inter
        topk[i] = (inter / uni).mean()

    order = np.argsort(topk)  # baixo → alto overlap
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    test_idx = order[:n_test]
    val_idx = order[n_test : n_test + n_val]
    train_idx = order[n_test + n_val :]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [15]:
def semantic_cluster_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    k: int | None = None,
    seed: int = 42,
) -> DatasetDict:
    """
    Clusters SBERT → reserva clusters distantes para test/val.
    """
    if k is None:
        k = int(np.sqrt(len(dataset)))

    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    embeddings = sbert.encode(
        [" ".join(t) for t in tqdm(dataset["tokens"])],
        batch_size=64,
        show_progress_bar=False,
    )

    km = KMeans(n_clusters=k, random_state=seed, n_init=10).fit(embeddings)
    labels = km.labels_
    centroids = km.cluster_centers_

    global_center = embeddings.mean(0, keepdims=True)
    dists = pairwise_distances(centroids, global_center).flatten()

    clusters_sorted = np.argsort(-dists)  # mais distantes primeiro
    test_clusters, val_clusters = set(), set()
    total_test = total_val = 0
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    for c in clusters_sorted:
        size = np.sum(labels == c)
        if total_test < n_test:  # preenche primeiro o teste
            test_clusters.add(c)
            total_test += size
        elif total_val < n_val:  # depois a validação
            val_clusters.add(c)
            total_val += size
        if total_test >= n_test and total_val >= n_val:
            break

    test_idx = np.where([lbl in test_clusters for lbl in labels])[0]
    val_idx = np.where([lbl in val_clusters for lbl in labels])[0]
    train_idx = np.where(
        [lbl not in test_clusters and lbl not in val_clusters for lbl in labels]
    )[0]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [16]:
def difficulty_scores(dataset: Dataset) -> np.ndarray:
    length = np.array([len(tok) for tok in dataset["tokens"]], dtype=float)
    length = (length - length.mean()) / length.std()

    dens = []
    for labels in dataset["ner_tags"]:
        non_o = sum(1 for l in labels if l != "O")
        dens.append(non_o / len(labels))
    dens = np.array(dens)
    dens = (dens - dens.mean()) / dens.std()

    ent_types, freq = [], Counter()
    for labels in dataset["ner_tags"]:
        types = [l[2:] for l in labels if l != "O"]
        ent_types.append(types[0] if types else "NONE")
    freq.update(ent_types)
    rarity = np.array([1 / freq[t] for t in ent_types])
    rarity = (rarity - rarity.mean()) / rarity.std()

    return length + dens + rarity


def reverse_curriculum_split(
    dataset: Dataset, pct_test: float = 0.20, pct_val: float = 0.10, seed: int = 42
) -> DatasetDict:
    scores = difficulty_scores(dataset)
    order = np.argsort(scores)  # easy→hard
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    test_idx = order[-n_test:]  # hardest
    val_idx = order[-(n_test + n_val) : -n_test]
    train_idx = order[: -(n_test + n_val)]

    rng = np.random.RandomState(seed)
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [17]:
def heur_len_split(dataset: Dataset,
                   pct_test: float = 0.20,
                   pct_val : float = 0.10,
                   seed: int = 42) -> DatasetDict:
    """Testa só sentenças ≥ máx(len_train)."""
    sent_lens = np.array([len(t) for t in dataset["tokens"]])
    # separação inicial train/dev (randômica estratificando por tamanho grosso)
    idx_all   = np.arange(len(dataset))
    train_idx, temp_idx = train_test_split(idx_all,
                                           test_size=pct_test + pct_val,
                                           stratify=(sent_lens//5),  # bin len
                                           random_state=seed)
    # define longo-threshold como tamanho máx. do treino
    max_train_len = sent_lens[train_idx].max()
    # test = sentenças > threshold.  Caso falte/ sobre exemplos, ajusta.
    test_idx  = [i for i in temp_idx if sent_lens[i] > max_train_len]
    resto_idx = [i for i in temp_idx if i not in test_idx]
    # completa ou reduz para atingir pct_test
    need = int(pct_test*len(dataset)) - len(test_idx)
    if need > 0:
        test_idx.extend(resto_idx[:need])
        val_idx = resto_idx[need:]
    else:
        val_keep = int(pct_val*len(dataset))
        val_idx  = resto_idx[:val_keep]
        test_idx = test_idx[: int(pct_test*len(dataset))]
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 2) Heuristic Rare-Words ----------------------------------
def heur_rare_split(dataset: Dataset,
                    pct_test: float = 0.20,
                    pct_val : float = 0.10,
                    seed: int = 42) -> DatasetDict:
    """Testa frases que contenham palavras do quintil + raro."""
    # contagem de frequência de token
    freqs = Counter(w for toks in dataset["tokens"] for w in toks)
    # define rareza: 20 % mais raras
    thresh = np.quantile(list(freqs.values()), 0.20)
    rare_set = {w for w,c in freqs.items() if c <= thresh}
    is_rare = np.array([any(w in rare_set for w in toks)
                        for toks in dataset["tokens"]])
    rare_idx   = np.where(is_rare)[0]
    common_idx = np.where(~is_rare)[0]
    # garante proporções desejadas
    n_test = int(pct_test*len(dataset))
    n_val  = int(pct_val *len(dataset))
    rng = np.random.default_rng(seed)
    test_idx = rng.choice(rare_idx, size=min(len(rare_idx), n_test),
                          replace=False)
    resto_idx = [i for i in rare_idx if i not in test_idx] + list(common_idx)
    val_idx  = rng.choice(resto_idx, size=n_val, replace=False)
    train_idx = [i for i in resto_idx if i not in val_idx]
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 3) Standard (80-10-10) -----------------------------------
def std_split(dataset: Dataset,
              pct_test: float = 0.10,
              pct_val : float = 0.10,
              seed: int = 42) -> DatasetDict:
    """Split aleatório estratificado por comprimento (PTB-like)."""
    idx = np.arange(len(dataset))
    strat = (np.array([len(t) for t in dataset["tokens"]]) // 5)
    train_idx, temp_idx = train_test_split(idx, test_size=pct_test+pct_val,
                                          stratify=strat, random_state=seed)
    val_rel = pct_val / (pct_test+pct_val)
    val_idx, test_idx = train_test_split(temp_idx, test_size=1-val_rel,
                                         stratify=strat[temp_idx],
                                         random_state=seed)
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 4) Adversarial (approx. Wasserstein) ---------------------
def adversarial_split(dataset: Dataset,
                      pct_test: float = 0.20,
                      pct_val : float = 0.10,
                      k: int = 5,
                      seed: int = 42) -> DatasetDict:
    """
    Seleciona k frases 'mais distantes' recursivamente (BallTree + W₂)
    para compor o teste, lembrando Alg.-1 de Søgaard et al.【turn6file4】.
    """
    # SBERT embed (rápido na GPU / aceitável CPU)
    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    emb = sbert.encode([" ".join(toks) for toks in dataset["tokens"]],
                       batch_size=64, show_progress_bar=False)
    # BallTree para vizinhança eficiente
    tree = BallTree(emb, leaf_size=40)
    idx_pool = set(range(len(dataset)))
    test_idx = []
    rng = np.random.default_rng(seed)
    print(f"Selecionando {int(pct_test*len(dataset))} sentenças para teste...")
    while len(test_idx) < int(pct_test*len(dataset)):
        print(f"  {len(test_idx)} selecionadas...")
        # amostra candidata: ponto + longe do centro
        center = emb[list(idx_pool)].mean(0, keepdims=True)
        dists, _ = tree.query(center, k=len(idx_pool))
        farthest = list(idx_pool)[int(dists.argmax())]
        # pega-se farest e seus k-NN mais próximos  ⇒ aumenta diversidade
        nn = tree.query([emb[farthest]], k=k, return_distance=False)[0]
        for j in nn:
            if j in idx_pool and len(test_idx) < int(pct_test*len(dataset)):
                test_idx.append(j)
                idx_pool.remove(j)
    # retira val
    val_size = int(pct_val*len(dataset))
    val_idx  = rng.choice(list(idx_pool), size=val_size, replace=False)
    idx_pool -= set(val_idx)
    train_idx = list(idx_pool)
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [18]:
def heur_len_split(dataset: Dataset,
                   pct_test: float = 0.20,
                   pct_val : float = 0.10,
                   seed: int = 42,
                   bin_size: int = 10) -> DatasetDict:
    """
    Teste = sentenças mais longas que max(len(train)).
    Robusto a datasets pequenos: se estratificação falhar, usa split aleatório.
    """
    rng = np.random.default_rng(seed)
    idx_all   = np.arange(len(dataset))
    sent_lens = np.array([len(t) for t in dataset["tokens"]])

    # ------ 1) tenta split estratificado por baldes -------------------------
    strat = (sent_lens // bin_size)
    try:
        train_idx, temp_idx = train_test_split(
            idx_all,
            test_size=pct_test + pct_val,
            stratify=strat,
            random_state=seed,
        )
    except ValueError:                       # classes com 1 amostra
        train_idx, temp_idx = train_test_split(
            idx_all,
            test_size=pct_test + pct_val,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    # ------ 2) escolhe test = len > max(train) ------------------------------
    max_train_len = sent_lens[train_idx].max()
    test_idx  = [i for i in temp_idx if sent_lens[i] > max_train_len]
    resto_idx = [i for i in temp_idx if i not in test_idx]

    # garante tamanhos exatos
    n_test_desired = int(pct_test * len(dataset))
    n_val_desired  = int(pct_val  * len(dataset))

    # completa teste se ficou pequeno
    if len(test_idx) < n_test_desired:
        extra = rng.choice(resto_idx,
                           size=n_test_desired - len(test_idx),
                           replace=False)
        test_idx.extend(extra)
        resto_idx = [i for i in resto_idx if i not in extra]

    # define validação
    val_idx  = rng.choice(resto_idx, size=n_val_desired, replace=False)
    train_idx = [i for i in idx_all if i not in test_idx and i not in val_idx]

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [19]:
def std_split(dataset,
              pct_test: float = 0.10,
              pct_val : float = 0.10,
              seed: int = 42,
              bin_size: int = 5) -> DatasetDict:
    """
    Split 80-10-10 robusto.
      • Tenta estratificar por comprimento // bin_size.
      • Se houver classes com <2 amostras, recua p/ split aleatório.
    """
    idx  = np.arange(len(dataset))
    bins = (np.array([len(t) for t in dataset["tokens"]]) // bin_size)

    try:
        train_idx, temp_idx = train_test_split(
            idx,
            test_size=pct_test + pct_val,
            stratify=bins,
            random_state=seed,
        )
    except ValueError:                       # classes muito pequenas
        train_idx, temp_idx = train_test_split(
            idx,
            test_size=pct_test + pct_val,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    # fraciona temp em val / test mantendo proporção desejada
    val_share = pct_val / (pct_test + pct_val)
    try:
        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=1 - val_share,
            stratify=bins[temp_idx],
            random_state=seed,
        )
    except ValueError:
        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=1 - val_share,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [20]:
def adversarial_split(dataset: Dataset,
                      pct_test: float = 0.20,
                      pct_val : float = 0.10,
                      k: int = 5,
                      seed: int = 42) -> DatasetDict:
    """
    Versão robusta: nunca “trava” antes de atingir n_test.
    Seleciona blocos de k sentenças mais distantes do centro iterativamente.
    """
    # -------------------------------- embeds -------------------------------
    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    emb = sbert.encode([" ".join(t) for t in dataset["tokens"]],
                       batch_size=64, show_progress_bar=False)
    tree = BallTree(emb, leaf_size=40)

    n_test = int(pct_test * len(dataset))
    n_val  = int(pct_val  * len(dataset))

    idx_pool = set(range(len(dataset)))
    test_idx = []
    rng = np.random.default_rng(seed)

    print(f"Selecionando {n_test} sentenças para teste…")
    step = 0
    while len(test_idx) < n_test and idx_pool:
        if step % 5 == 0:
            print(f"  {len(test_idx)} selecionadas…")
        step += 1

        pool_list = list(idx_pool)
        center = emb[pool_list].mean(0, keepdims=True)

        # distância euclidiana ao centro global restante
        dists, _ = tree.query(center, k=len(pool_list))
        farthest_local_idx = int(dists.argmax())
        farthest_global_idx = pool_list[farthest_local_idx]

        # número efetivo de vizinhos
        k_eff = min(k, len(idx_pool))
        nn = set(tree.query([emb[farthest_global_idx]],
                            k=k_eff,
                            return_distance=False)[0])

        # adiciona vizinhos ainda não selecionados
        for j in nn:
            if j in idx_pool and len(test_idx) < n_test:
                test_idx.append(j)
                idx_pool.remove(j)

        # Se nada foi adicionado (pode acontecer quando sobram <k únicos)
        if farthest_global_idx not in test_idx:
            test_idx.append(farthest_global_idx)
            idx_pool.remove(farthest_global_idx)

    # ------------------------ validação e treino ---------------------------
    val_idx = rng.choice(list(idx_pool), size=n_val, replace=False)
    idx_pool -= set(val_idx)
    train_idx = list(idx_pool)

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [21]:
# standard_split = std_split(cachaca_full)
# print('std')
# # random_splt = random_splits(cachaca_full)
# # print('random')
# heur_len = heur_len_split(cachaca_full)
# print("heur_len")
# heur_rare = heur_rare_split(cachaca_full)
# print("heur_rare")
# advers = adversarial_split(cachaca_full)
# print("advs")
# loc = loc_split(cachaca_full)
# print("loc")
# semantic = semantic_cluster_split(cachaca_full)
# print("semantic")
# reverse = reverse_curriculum_split(cachaca_full)
# print("reverse")

# Experimentos

In [22]:

from sklearn.metrics import f1_score as skl_f1

In [23]:
def train_ner_with_split(
    dataset: Dataset,
    split: str,  # "loc" | "semantic" | "reverse" | func
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    model_ckpt: str = "Davlan/distilbert-base-multilingual-cased-ner-hrl",
    training_args_kwargs: dict | None = None,
    split_kwargs: dict | None = None,
    seed: int = 42,
):
    """
    Treina um modelo de NER usando a estratégia de split desejada.
    Retorna (trainer, métricas_test).
    """
    # 1. Escolhe função de split ------------------------------------------------
    if callable(split):
        split_fn = split
    else:
        _map = {
            "loc"      : loc_split,
            "reverse"  : reverse_curriculum_split,
            "semantic" : semantic_cluster_split,
            "heur_len" : heur_len_split,
            "heur_rare": heur_rare_split,
            "std"      : std_split,
            "advs"     : adversarial_split,
        }
        if split not in _map:
            raise ValueError(f"split='{split}' não reconhecido.")
        split_fn = _map[split]

    split_kwargs = split_kwargs or {}
    ds = split_fn(
        dataset, pct_test=pct_test, pct_val=pct_val, seed=seed, **split_kwargs
    )  # train/val/test

    # 2. Tokenizer e modelo -----------------------------------------------------
    label_list = sorted({l for labels in dataset["ner_tags"] for l in labels})
    label2id = {l: i for i, l in enumerate(label_list)}
    id2label = {i: l for l, i in label2id.items()}

    num_labels = len(label_list)
    tok = AutoTokenizer.from_pretrained(model_ckpt)
    model = AutoModelForTokenClassification.from_pretrained(
        model_ckpt,
        num_labels=num_labels,  # ← adapta o tamanho
        ignore_mismatched_sizes=True,  # ← descarta pesos velhos da head
    )
    model.config.id2label = id2label
    model.config.label2id = label2id

    # 3. Mapeamento label↔id ----------------------------------------------------
    # label_list = sorted(
    #     {l for labels in dataset["ner_tags"] for l in labels if l != "O"}
    # )

    # 4. Tokenização + alinhamento ---------------------------------------------
    # def tok_function(ex):
    #     return tok(
    #         ex["tokens"], is_split_into_words=True, truncation=True, padding=False
    #     )

    # def align_labels(ex):
    #     word_ids = ex.word_ids()
    #     labels = []
    #     for w in word_ids:
    #         if w is None:
    #             labels.append(-100)
    #         else:
    #             labels.append(label2id.get(ex["ner_tags"][w], 0))
    #     ex["labels"] = labels
    #     return ex

    # ds_tok = ds.map(tok_function, batched=True)
    # ds_tok = ds_tok.map(align_labels)

    def tokenize_and_align_labels(examples, label_all_tokens=False):
        tokenized = tok(examples["tokens"], is_split_into_words=True, truncation=True)

        labels_batch = []
        for i, word_labels in enumerate(examples["ner_tags"]):
            word_ids = tokenized.word_ids(batch_index=i)  # <- aqui sim
            label_ids = []
            previous_word_idx = None
            for word_idx in word_ids:
                if word_idx is None:
                    label_ids.append(-100)  # máscara
                elif word_idx != previous_word_idx:
                    label_ids.append(label2id[word_labels[word_idx]])
                else:
                    # marca sub-tokens; mude para `label2id[...]`
                    # se quiser repetir label em todos os sub-tokens
                    label_ids.append(
                        label2id[word_labels[word_idx]] if label_all_tokens else -100
                    )
                previous_word_idx = word_idx
            labels_batch.append(label_ids)

        tokenized["labels"] = labels_batch
        return tokenized

    ds_tok = ds.map(
        tokenize_and_align_labels, batched=True, remove_columns=ds["train"].column_names
    )
    # 5. Métrica (seqeval) ------------------------------------------------------
    seqeval = load_metric("seqeval")

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)

        sent_preds, sent_labels = [], []   # p/ seqeval
        flat_preds, flat_labels = [], []   # p/ sklearn

        for p_row, l_row in zip(preds, labels):
            p_sent, l_sent = [], []
            for pi, li in zip(p_row, l_row):
                if li != -100:
                    lbl_true = id2label[li]
                    lbl_pred = id2label[pi]
                    p_sent.append(lbl_pred)
                    l_sent.append(lbl_true)
                    flat_preds.append(lbl_pred)
                    flat_labels.append(lbl_true)
            sent_preds.append(p_sent)
            sent_labels.append(l_sent)

        # métricas seqeval (micro F1 = overall_f1)
        seqeval_metrics = seqeval.compute(
            predictions=sent_preds,
            references=sent_labels,
        )

        # métricas sklearn
        f1_micro    = skl_f1(flat_labels, flat_preds, average="micro",    zero_division=0)
        f1_macro    = skl_f1(flat_labels, flat_preds, average="macro",    zero_division=0)
        f1_weighted = skl_f1(flat_labels, flat_preds, average="weighted", zero_division=0)

        return {
            **seqeval_metrics,            # overall_precision / recall / f1
            "f1_micro":    f1_micro,
            "f1_macro":    f1_macro,
            "f1_weighted": f1_weighted,
        }



    # 6. Args de treinamento ----------------------------------------------------
    args_defaults = dict(
        output_dir=f"ner-{split}",
        # estratégia de avaliação + salvamento
        eval_strategy="epoch",  # novo nome (4.52+)
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="overall_f1",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=5,
        seed=seed,
        report_to="none"
    )

    if training_args_kwargs:
        args_defaults.update(training_args_kwargs)
    args = TrainingArguments(**args_defaults)

    data_collator = DataCollatorForTokenClassification(tokenizer=tok, padding=True)

    # 7. Trainer ---------------------------------------------------------------
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=ds_tok["train"],
        eval_dataset=ds_tok["val"],
        tokenizer=tok,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    # 8. Avaliação final --------------------------------------------------------
    test_metrics = trainer.evaluate(eval_dataset=ds_tok["test"])
    return trainer, test_metrics

In [24]:
#splits = ["loc", "reverse", "semantic", "heur_len", "heur_rare", "std", "advs"]

In [ ]:
results = {}
trainer_all = {}
for s in splits:
    print(f"Treinando com split: {s}")
    trainer, metrics = train_ner_with_split(cachaca_full, split=s)
    results[s] = metrics
    trainer_all[s] = trainer
    print("F1 Macro:", metrics["eval_f1_macro"])
    print("F1 Micro:", metrics["eval_f1_micro"])
    print("F1 Weighted:", metrics["eval_f1_weighted"])
    print(metrics)
    print('\n')

Treinando com split: loc


C:\Users\user\AppData\Local\Temp\ipykernel_13776\4053031328.py:28: RuntimeWarning: invalid value encountered in divide
  topk[i] = (inter / uni).mean()
Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([35]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([35, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 2725/2725 [00:00<00:00, 9894.77 examples/s] 
C:\Users\user\AppData\Local\Temp\ipykernel_13776\2810120630.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
c:

Epoch,Training Loss,Validation Loss,Caracteristica Sensorial Aroma,Caracteristica Sensorial Consistência,Caracteristica Sensorial Cor,Caracteristica Sensorial Sabor,Classificacao Bebida,Equipamento Destilacao,Graduacao Alcoolica,Nome Bebida,Nome Local,Nome Organizacao,Nome Pessoa,Recipiente Armazenamento,Tempo,Tempo Armazenamento,Tipo Madeira,Volume,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.230700,0.148722,"{'precision': 0.56, 'recall': 0.5833333333333334, 'f1': 0.5714285714285714, 'number': 120}","{'precision': 0.8055555555555556, 'recall': 0.6590909090909091, 'f1': 0.7250000000000001, 'number': 44}","{'precision': 0.6436781609195402, 'recall': 0.7, 'f1': 0.6706586826347305, 'number': 80}","{'precision': 0.5568181818181818, 'recall': 0.392, 'f1': 0.460093896713615, 'number': 125}","{'precision': 0.8841463414634146, 'recall': 0.8529411764705882, 'f1': 0.8682634730538923, 'number': 170}","{'precision': 0.7567567567567568, 'recall': 0.5833333333333334, 'f1': 0.6588235294117648, 'number': 48}","{'precision': 0.7666666666666667, 'recall': 0.8679245283018868, 'f1': 0.8141592920353983, 'number': 53}","{'precision': 0.7388535031847133, 'recall': 0.7341772151898734, 'f1': 0.7365079365079364, 'number': 474}","{'precision': 0.8421955403087479, 'recall': 0.9229323308270677, 'f1': 0.8807174887892377, 'number': 532}","{'precision': 0.5660377358490566, 'recall': 0.625, 'f1': 0.594059405940594, 'number': 144}","{'precision': 0.868020304568528, 'recall': 0.8860103626943006, 'f1': 0.876923076923077, 'number': 193}","{'precision': 0.9539473684210527, 'recall': 0.8682634730538922, 'f1': 0.9090909090909091, 'number': 167}","{'precision': 0.8636363636363636, 'recall': 0.9047619047619048, 'f1': 0.8837209302325582, 'number': 189}","{'precision': 0.9071428571428571, 'recall': 0.9548872180451128, 'f1': 0.9304029304029303, 'number': 133}","{'precision': 0.9252669039145908, 'recall': 0.8873720136518771, 'f1': 0.9059233449477352, 'number': 293}","{'precision': 0.9723756906077348, 'recall': 0.9887640449438202, 'f1': 0.9805013927576601, 'number': 178}",0.811761,0.816174,0.813961,0.963127,0.963127,0.788513,0.962038
2,0.054500,0.149577,"{'precision': 0.48484848484848486, 'recall': 0.6666666666666666, 'f1': 0.5614035087719298, 'number': 120}","{'precision': 0.8823529411764706, 'recall': 0.6818181818181818, 'f1': 0.7692307692307693, 'number': 44}","{'precision': 0.7073170731707317, 'recall': 0.725, 'f1': 0.7160493827160495, 'number': 80}","{'precision': 0.5168539325842697, 'recall': 0.368, 'f1': 0.4299065420560748, 'number': 125}","{'precision': 0.842391304347826, 'recall': 0.9117647058823529, 'f1': 0.8757062146892656, 'number': 170}","{'precision': 0.7719298245614035, 'recall': 0.9166666666666666, 'f1': 0.838095238095238, 'number': 48}","{'precision': 0.8103448275862069, 'recall': 0.8867924528301887, 'f1': 0.8468468468468469, 'number': 53}","{'precision': 0.7634854771784232, 'recall': 0.7763713080168776, 'f1': 0.7698744769874477, 'number': 474}","{'precision': 0.8470790378006873, 'recall': 0.9266917293233082, 'f1': 0.8850987432675045, 'number': 532}","{'precision': 0.5153061224489796, 'recall': 0.7013888888888888, 'f1': 0.5941176470588234, 'number': 144}","{'precision': 0.8702702702702703, 'recall': 0.8341968911917098, 'f1': 0.851851851851852, 'number': 193}","{'precision': 0.927710843373494, 'recall': 0.9221556886227545, 'f1': 0.924924924924925, 'number': 167}","{'precision': 0.8704663212435233, 'recall': 0.8888888888888888, 'f1': 0.8795811518324607, 'number': 189}","{'precision': 0.9703703703703703, 'recall': 0.9849624060150376, 'f1': 0.9776119402985074, 'number': 133}","{'precision': 0.8532110091743119, 'recall': 0.9522184300341296, 'f1': 0.9, 'number': 293}","{'precision': 0.9672131147540983, 'recall': 0.9943820224719101, 'f1': 0.9806094182825486, 'number': 178}",0.799230,0.846755,0.822307,0.964653,0.964653,0.802661,0.964358
3,0.035900,0.138320,"{'precision': 0.525, 'recall': 0.7, 'f1': 0.6, 'number': 1

c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warn

c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


...

c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


F1 Macro: 0.6963812336091685
F1 Micro: 0.7734393085886256
F1 Weighted: 0.6859838994546551
{'eval_loss': 1.9648122787475586, 'eval_CARACTERISTICA_SENSORIAL_CONSISTÊNCIA': {'precision': 1.0, 'recall': 0.82, 'f1': 0.9010989010989011, 'number': 50}, 'eval_CARACTERISTICA_SENSORIAL_SABOR': {'precision': 0.75, 'recall': 0.8709677419354839, 'f1': 0.8059701492537312, 'number': 31}, 'eval_CLASSIFICACAO_BEBIDA': {'precision': 0.9, 'recall': 0.5294117647058824, 'f1': 0.6666666666666667, 'number': 17}, 'eval_GRADUACAO_ALCOOLICA': {'precision': 0.9746192893401016, 'recall': 0.9922480620155039, 'f1': 0.9833546734955186, 'number': 387}, 'eval_NOME_BEBIDA': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 12}, 'eval_NOME_LOCAL': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 244}, 'eval_PRECO': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 710}, 'eval_RECIPIENTE_ARMAZENAMENTO': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 0}, 'eval_TEMPO': {'precision': 0.97058823529411

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([35]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([35, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 2725/2725 [00:00<00:00, 9038.92 examples/s] 
C:\Users\user\AppData\Local\Temp\ipykernel_13776\2810120630.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as 

Epoch,Training Loss,Validation Loss,Caracteristica Sensorial Aroma,Caracteristica Sensorial Consistência,Caracteristica Sensorial Cor,Caracteristica Sensorial Sabor,Classificacao Bebida,Equipamento Destilacao,Graduacao Alcoolica,Nome Bebida,Nome Local,Nome Organizacao,Nome Pessoa,Preco,Recipiente Armazenamento,Tempo,Tempo Armazenamento,Tipo Madeira,Volume,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.253500,0.087379,"{'precision': 0.7009345794392523, 'recall': 0.8522727272727273, 'f1': 0.7692307692307693, 'number': 88}","{'precision': 1.0, 'recall': 0.7419354838709677, 'f1': 0.8518518518518519, 'number': 31}","{'precision': 0.782608695652174, 'recall': 0.6792452830188679, 'f1': 0.7272727272727273, 'number': 53}","{'precision': 0.5616438356164384, 'recall': 0.5774647887323944, 'f1': 0.5694444444444443, 'number': 71}","{'precision': 0.84472049689441, 'recall': 0.9379310344827586, 'f1': 0.888888888888889, 'number': 145}","{'precision': 0.78125, 'recall': 1.0, 'f1': 0.8771929824561403, 'number': 25}","{'precision': 0.9523809523809523, 'recall': 0.9917355371900827, 'f1': 0.97165991902834, 'number': 121}","{'precision': 0.8472622478386167, 'recall': 0.9158878504672897, 'f1': 0.8802395209580838, 'number': 321}","{'precision': 0.9671052631578947, 'recall': 0.9483870967741935, 'f1': 0.9576547231270358, 'number': 465}","{'precision': 0.7288135593220338, 'recall': 0.7818181818181819, 'f1': 0.7543859649122807, 'number': 110}","{'precision': 0.8645833333333334, 'recall': 0.83, 'f1': 0.8469387755102041, 'number': 100}","{'precision': 0.9642857142857143, 'recall': 1.0, 'f1': 0.9818181818181818, 'number': 81}","{'precision': 0.9292929292929293, 'recall': 0.9583333333333334, 'f1': 0.9435897435897437, 'number': 96}","{'precision': 0.935672514619883, 'recall': 0.9411764705882353, 'f1': 0.93841642228739, 'number': 170}","{'precision': 0.9478260869565217, 'recall': 0.990909090909091, 'f1': 0.9688888888888889, 'number': 110}","{'precision': 0.8888888888888888, 'recall': 0.9451476793248945, 'f1': 0.9161554192229039, 'number': 237}","{'precision': 0.9787234042553191, 'recall': 0.9829059829059829, 'f1': 0.9808102345415778, 'number': 234}",0.887839,0.917819,0.902581,0.975946,0.975946,0.860786,0.975884
2,0.068900,0.077845,"{'precision': 0.7378640776699029, 'recall': 0.8636363636363636, 'f1': 0.7958115183246073, 'number': 88}","{'precision': 0.96, 'recall': 0.7741935483870968, 'f1': 0.8571428571428571, 'number': 31}","{'precision': 0.7115384615384616, 'recall': 0.6981132075471698, 'f1': 0.7047619047619047, 'number': 53}","{'precision': 0.7076923076923077, 'recall': 0.647887323943662, 'f1': 0.676470588235294, 'number': 71}","{'precision': 0.8766233766233766, 'recall': 0.9310344827586207, 'f1': 0.9030100334448159, 'number': 145}","{'precision': 0.6944444444444444, 'recall': 1.0, 'f1': 0.819672131147541, 'number': 25}","{'precision': 0.967741935483871, 'recall': 0.9917355371900827, 'f1': 0.979591836734694, 'number': 121}","{'precision': 0.8769716088328076, 'recall': 0.8660436137071651, 'f1': 0.8714733542319749, 'number': 321}","{'precision': 0.9574468085106383, 'recall': 0.967741935483871, 'f1': 0.9625668449197862, 'number': 465}","{'precision': 0.7647058823529411, 'recall': 0.8272727272727273, 'f1': 0.794759825327511, 'number': 110}","{'precision': 0.90625, 'recall': 0.87, 'f1': 0.8877551020408163, 'number': 100}","{'precision': 0.9642857142857143, 'recall': 1.0, 'f1': 0.9818181818181818, 'number': 81}","{'precision': 0.9393939393939394, 'recall': 0.96875, 'f1': 0.9538461538461539, 'number': 96}","{'precision': 0.9310344827586207, 'recall': 0.9529411764705882, 'f1': 0.941860465116279, 'number': 170}","{'precision': 0.9734513274336283, 'recall': 1.0, 'f1': 0.9865470852017937, 'number': 110}","{'precision': 0.950207468879668, 'recall': 0.9662447257383966, 'f1': 0.9581589958158996, 'number': 237}","{'precision': 0.9829059829059829, 'recall': 0.9829059829059829, 'f1': 0.9829059829059829, 'number': 234}",0.907422

c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warn

In [32]:
splits_att = ["heur_len", "heur_rare", "std", "advs"]

In [ ]:
results = {}
trainer_all = {}
for s in splits_att:
    print(f"Treinando com split: {s}")
    trainer, metrics = train_ner_with_split(cachaca_full, split=s)
    results[s] = metrics
    trainer_all[s] = trainer
    
    print("Trainer vai usar:", trainer.args.device)
    print("F1 Macro:", metrics["eval_f1_macro"])
    print("F1 Micro:", metrics["eval_f1_micro"])
    print("F1 Weighted:", metrics["eval_f1_weighted"])
    print(metrics)
    print('\n')

Treinando com split: heur_len


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([35]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([35, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 2725/2725 [00:00<00:00, 16005.41 examples/s]
C:\Users\user\AppData\Local\Temp\ipykernel_5540\2810120630.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Caracteristica Sensorial Aroma,Caracteristica Sensorial Consistência,Caracteristica Sensorial Cor,Caracteristica Sensorial Sabor,Classificacao Bebida,Equipamento Destilacao,Graduacao Alcoolica,Nome Bebida,Nome Local,Nome Organizacao,Nome Pessoa,Preco,Recipiente Armazenamento,Tempo,Tempo Armazenamento,Tipo Madeira,Volume,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.258100,0.086862,"{'precision': 0.7326732673267327, 'recall': 0.8409090909090909, 'f1': 0.783068783068783, 'number': 88}","{'precision': 1.0, 'recall': 0.7419354838709677, 'f1': 0.8518518518518519, 'number': 31}","{'precision': 0.6862745098039216, 'recall': 0.660377358490566, 'f1': 0.6730769230769231, 'number': 53}","{'precision': 0.6031746031746031, 'recall': 0.5352112676056338, 'f1': 0.5671641791044777, 'number': 71}","{'precision': 0.8058823529411765, 'recall': 0.9448275862068966, 'f1': 0.8698412698412699, 'number': 145}","{'precision': 0.8846153846153846, 'recall': 0.92, 'f1': 0.9019607843137256, 'number': 25}","{'precision': 0.96, 'recall': 0.9917355371900827, 'f1': 0.975609756097561, 'number': 121}","{'precision': 0.8210227272727273, 'recall': 0.9003115264797508, 'f1': 0.8588410104011887, 'number': 321}","{'precision': 0.9573560767590619, 'recall': 0.9655913978494624, 'f1': 0.961456102783726, 'number': 465}","{'precision': 0.7850467289719626, 'recall': 0.7636363636363637, 'f1': 0.7741935483870966, 'number': 110}","{'precision': 0.8446601941747572, 'recall': 0.87, 'f1': 0.8571428571428571, 'number': 100}","{'precision': 0.9642857142857143, 'recall': 1.0, 'f1': 0.9818181818181818, 'number': 81}","{'precision': 0.9285714285714286, 'recall': 0.9479166666666666, 'f1': 0.9381443298969072, 'number': 96}","{'precision': 0.9523809523809523, 'recall': 0.9411764705882353, 'f1': 0.9467455621301775, 'number': 170}","{'precision': 0.9478260869565217, 'recall': 0.990909090909091, 'f1': 0.9688888888888889, 'number': 110}","{'precision': 0.902834008097166, 'recall': 0.9409282700421941, 'f1': 0.9214876033057852, 'number': 237}","{'precision': 0.9871244635193133, 'recall': 0.9829059829059829, 'f1': 0.9850107066381155, 'number': 234}",0.888757,0.916599,0.902463,0.976312,0.976312,0.860720,0.976000
2,0.070400,0.072647,"{'precision': 0.8588235294117647, 'recall': 0.8295454545454546, 'f1': 0.8439306358381502, 'number': 88}","{'precision': 0.8888888888888888, 'recall': 0.7741935483870968, 'f1': 0.8275862068965517, 'number': 31}","{'precision': 0.7254901960784313, 'recall': 0.6981132075471698, 'f1': 0.7115384615384615, 'number': 53}","{'precision': 0.71875, 'recall': 0.647887323943662, 'f1': 0.6814814814814815, 'number': 71}","{'precision': 0.9090909090909091, 'recall': 0.9655172413793104, 'f1': 0.9364548494983278, 'number': 145}","{'precision': 0.7352941176470589, 'recall': 1.0, 'f1': 0.8474576271186441, 'number': 25}","{'precision': 0.9836065573770492, 'recall': 0.9917355371900827, 'f1': 0.9876543209876544, 'number': 121}","{'precision': 0.8926380368098159, 'recall': 0.9065420560747663, 'f1': 0.8995363214837713, 'number': 321}","{'precision': 0.9597457627118644, 'recall': 0.9741935483870968, 'f1': 0.966915688367129, 'number': 465}","{'precision': 0.8348623853211009, 'recall': 0.8272727272727273, 'f1': 0.8310502283105023, 'number': 110}","{'precision': 0.898989898989899, 'recall': 0.89, 'f1': 0.8944723618090452, 'number': 100}","{'precision': 0.9642857142857143, 'recall': 1.0, 'f1': 0.9818181818181818, 'number': 81}","{'precision': 0.9587628865979382, 'recall': 0.96875, 'f1': 0.9637305699481865, 'number': 96}","{'precision': 0.9421965317919075, 'recall': 0.9588235294117647, 'f1': 0.9504373177842566, 'number': 170}","{'precision': 0.9734513274336283, 'recall': 1.0, 'f1': 0.9865470852017937, 'number': 110}","{'precision': 0.9416666666666667, 'recall': 0.9535864978902954, 'f1': 0.9475890985324946, 'number': 237}","{'precision': 0.9913793103448276, 'recall': 0.9829059829059829, 'f1': 0.9871244635193134, 'n

Trainer vai usar: cuda:0
F1 Macro: 0.9276999297146507
F1 Micro: 0.9850684931506849
F1 Weighted: 0.9850116789596302
{'eval_loss': 0.0642155259847641, 'eval_CARACTERISTICA_SENSORIAL_AROMA': {'precision': 0.7751196172248804, 'recall': 0.826530612244898, 'f1': 0.7999999999999999, 'number': 196}, 'eval_CARACTERISTICA_SENSORIAL_CONSISTÊNCIA': {'precision': 0.92, 'recall': 0.8846153846153846, 'f1': 0.9019607843137256, 'number': 52}, 'eval_CARACTERISTICA_SENSORIAL_COR': {'precision': 0.9243697478991597, 'recall': 0.9243697478991597, 'f1': 0.9243697478991597, 'number': 119}, 'eval_CARACTERISTICA_SENSORIAL_SABOR': {'precision': 0.7898089171974523, 'recall': 0.7085714285714285, 'f1': 0.746987951807229, 'number': 175}, 'eval_CLASSIFICACAO_BEBIDA': {'precision': 0.9301470588235294, 'recall': 0.9656488549618321, 'f1': 0.947565543071161, 'number': 262}, 'eval_EQUIPAMENTO_DESTILACAO': {'precision': 0.9230769230769231, 'recall': 0.9230769230769231, 'f1': 0.9230769230769231, 'number': 65}, 'eval_GRADUAC

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([35]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([35, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 1729/1729 [00:00<00:00, 5096.03 examples/s]
C:\Users\user\AppData\Local\Temp\ipykernel_5540\2810120630.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Caracteristica Sensorial Aroma,Caracteristica Sensorial Consistência,Caracteristica Sensorial Cor,Caracteristica Sensorial Sabor,Classificacao Bebida,Equipamento Destilacao,Graduacao Alcoolica,Nome Bebida,Nome Local,Nome Organizacao,Nome Pessoa,Preco,Recipiente Armazenamento,Tempo,Tempo Armazenamento,Tipo Madeira,Volume,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.250800,0.066531,"{'precision': 0.6792452830188679, 'recall': 0.4444444444444444, 'f1': 0.5373134328358209, 'number': 81}","{'precision': 0.9629629629629629, 'recall': 0.7878787878787878, 'f1': 0.8666666666666665, 'number': 33}","{'precision': 0.9215686274509803, 'recall': 0.8245614035087719, 'f1': 0.8703703703703703, 'number': 57}","{'precision': 0.5043478260869565, 'recall': 0.7733333333333333, 'f1': 0.6105263157894737, 'number': 75}","{'precision': 0.9541984732824428, 'recall': 0.8741258741258742, 'f1': 0.9124087591240876, 'number': 143}","{'precision': 0.8333333333333334, 'recall': 0.8333333333333334, 'f1': 0.8333333333333334, 'number': 30}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 136}","{'precision': 0.8307692307692308, 'recall': 0.8737864077669902, 'f1': 0.8517350157728706, 'number': 309}","{'precision': 0.9549763033175356, 'recall': 0.96875, 'f1': 0.9618138424821002, 'number': 416}","{'precision': 0.8952380952380953, 'recall': 0.8545454545454545, 'f1': 0.8744186046511628, 'number': 110}","{'precision': 1.0, 'recall': 0.9565217391304348, 'f1': 0.9777777777777777, 'number': 69}","{'precision': 0.9772727272727273, 'recall': 1.0, 'f1': 0.9885057471264368, 'number': 86}","{'precision': 0.9506172839506173, 'recall': 0.9506172839506173, 'f1': 0.9506172839506173, 'number': 81}","{'precision': 0.9911504424778761, 'recall': 0.9911504424778761, 'f1': 0.9911504424778761, 'number': 113}","{'precision': 0.9646017699115044, 'recall': 0.990909090909091, 'f1': 0.9775784753363229, 'number': 110}","{'precision': 0.9591078066914498, 'recall': 0.9591078066914498, 'f1': 0.9591078066914497, 'number': 269}","{'precision': 1.0, 'recall': 0.9896193771626297, 'f1': 0.994782608695652, 'number': 289}",0.918291,0.919817,0.919054,0.981133,0.981133,0.878251,0.980556
2,0.065700,0.051191,"{'precision': 0.7738095238095238, 'recall': 0.8024691358024691, 'f1': 0.787878787878788, 'number': 81}","{'precision': 0.967741935483871, 'recall': 0.9090909090909091, 'f1': 0.9374999999999999, 'number': 33}","{'precision': 0.9433962264150944, 'recall': 0.8771929824561403, 'f1': 0.9090909090909091, 'number': 57}","{'precision': 0.7228915662650602, 'recall': 0.8, 'f1': 0.759493670886076, 'number': 75}","{'precision': 0.9779411764705882, 'recall': 0.9300699300699301, 'f1': 0.953405017921147, 'number': 143}","{'precision': 0.9615384615384616, 'recall': 0.8333333333333334, 'f1': 0.8928571428571429, 'number': 30}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 136}","{'precision': 0.8699690402476781, 'recall': 0.9093851132686084, 'f1': 0.8892405063291139, 'number': 309}","{'precision': 0.9830097087378641, 'recall': 0.9735576923076923, 'f1': 0.9782608695652173, 'number': 416}","{'precision': 0.9611650485436893, 'recall': 0.9, 'f1': 0.9295774647887324, 'number': 110}","{'precision': 1.0, 'recall': 0.9565217391304348, 'f1': 0.9777777777777777, 'number': 69}","{'precision': 0.9772727272727273, 'recall': 1.0, 'f1': 0.9885057471264368, 'number': 86}","{'precision': 0.9404761904761905, 'recall': 0.9753086419753086, 'f1': 0.9575757575757574, 'number': 81}","{'precision': 0.9823008849557522, 'recall': 0.9823008849557522, 'f1': 0.9823008849557522, 'number': 113}","{'precision': 0.9642857142857143, 'recall': 0.9818181818181818, 'f1': 0.972972972972973, 'number': 110}","{'precision': 0.967032967032967, 'recall': 0.9814126394052045, 'f1': 0.974169741697417, 'number': 269}","{'precision': 1.0, 'recall': 0.9896193771626297, 'f1': 0.994782608695652, 'number': 289}",0.948111,0.948899,0.948505,0.986891,0.986891,0.944242,0.9

Trainer vai usar: cuda:0
F1 Macro: 0.837931694141208
F1 Micro: 0.9647833016297362
F1 Weighted: 0.9646389998275015
{'eval_loss': 0.1483357846736908, 'eval_CARACTERISTICA_SENSORIAL_AROMA': {'precision': 0.6850393700787402, 'recall': 0.7341772151898734, 'f1': 0.7087576374745418, 'number': 237}, 'eval_CARACTERISTICA_SENSORIAL_CONSISTÊNCIA': {'precision': 0.7361111111111112, 'recall': 0.828125, 'f1': 0.7794117647058824, 'number': 64}, 'eval_CARACTERISTICA_SENSORIAL_COR': {'precision': 0.6822429906542056, 'recall': 0.7849462365591398, 'f1': 0.73, 'number': 93}, 'eval_CARACTERISTICA_SENSORIAL_SABOR': {'precision': 0.5944700460829493, 'recall': 0.5682819383259912, 'f1': 0.5810810810810811, 'number': 227}, 'eval_CLASSIFICACAO_BEBIDA': {'precision': 0.7666666666666667, 'recall': 0.9096045197740112, 'f1': 0.8320413436692506, 'number': 177}, 'eval_EQUIPAMENTO_DESTILACAO': {'precision': 0.8032786885245902, 'recall': 0.875, 'f1': 0.8376068376068376, 'number': 56}, 'eval_GRADUACAO_ALCOOLICA': {'preci

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([35]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([35, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 2727/2727 [00:00<00:00, 15937.10 examples/s]
C:\Users\user\AppData\Local\Temp\ipykernel_5540\2810120630.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Caracteristica Sensorial Aroma,Caracteristica Sensorial Consistência,Caracteristica Sensorial Cor,Caracteristica Sensorial Sabor,Classificacao Bebida,Equipamento Destilacao,Graduacao Alcoolica,Nome Bebida,Nome Local,Nome Organizacao,Nome Pessoa,Preco,Recipiente Armazenamento,Tempo,Tempo Armazenamento,Tipo Madeira,Volume,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.261500,0.094801,"{'precision': 0.6746031746031746, 'recall': 0.7589285714285714, 'f1': 0.7142857142857143, 'number': 112}","{'precision': 0.8620689655172413, 'recall': 0.6578947368421053, 'f1': 0.746268656716418, 'number': 38}","{'precision': 0.8070175438596491, 'recall': 0.8070175438596491, 'f1': 0.8070175438596491, 'number': 57}","{'precision': 0.6086956521739131, 'recall': 0.4827586206896552, 'f1': 0.5384615384615384, 'number': 87}","{'precision': 0.92, 'recall': 0.905511811023622, 'f1': 0.9126984126984127, 'number': 127}","{'precision': 0.88, 'recall': 0.8461538461538461, 'f1': 0.8627450980392156, 'number': 26}","{'precision': 0.9545454545454546, 'recall': 0.9722222222222222, 'f1': 0.963302752293578, 'number': 108}","{'precision': 0.8501440922190202, 'recall': 0.8676470588235294, 'f1': 0.8588064046579331, 'number': 340}","{'precision': 0.9530516431924883, 'recall': 0.9666666666666667, 'f1': 0.9598108747044918, 'number': 420}","{'precision': 0.7572815533980582, 'recall': 0.8041237113402062, 'f1': 0.78, 'number': 97}","{'precision': 0.8658536585365854, 'recall': 0.8987341772151899, 'f1': 0.8819875776397516, 'number': 79}","{'precision': 0.92, 'recall': 1.0, 'f1': 0.9583333333333334, 'number': 92}","{'precision': 0.9340659340659341, 'recall': 0.9770114942528736, 'f1': 0.9550561797752809, 'number': 87}","{'precision': 0.9590163934426229, 'recall': 0.936, 'f1': 0.9473684210526315, 'number': 125}","{'precision': 0.9186991869918699, 'recall': 1.0, 'f1': 0.9576271186440678, 'number': 113}","{'precision': 0.9523809523809523, 'recall': 0.9649122807017544, 'f1': 0.9586056644880173, 'number': 228}","{'precision': 0.9758064516129032, 'recall': 0.968, 'f1': 0.9718875502008033, 'number': 250}",0.894366,0.904862,0.899583,0.975140,0.975140,0.862612,0.974537
2,0.073800,0.083316,"{'precision': 0.6333333333333333, 'recall': 0.8482142857142857, 'f1': 0.7251908396946566, 'number': 112}","{'precision': 0.9117647058823529, 'recall': 0.8157894736842105, 'f1': 0.861111111111111, 'number': 38}","{'precision': 0.9259259259259259, 'recall': 0.8771929824561403, 'f1': 0.9009009009009009, 'number': 57}","{'precision': 0.8163265306122449, 'recall': 0.45977011494252873, 'f1': 0.5882352941176471, 'number': 87}","{'precision': 0.8823529411764706, 'recall': 0.9448818897637795, 'f1': 0.9125475285171102, 'number': 127}","{'precision': 0.96, 'recall': 0.9230769230769231, 'f1': 0.9411764705882353, 'number': 26}","{'precision': 0.9906542056074766, 'recall': 0.9814814814814815, 'f1': 0.986046511627907, 'number': 108}","{'precision': 0.8847262247838616, 'recall': 0.9029411764705882, 'f1': 0.8937409024745268, 'number': 340}","{'precision': 0.9829683698296837, 'recall': 0.9619047619047619, 'f1': 0.9723225030084236, 'number': 420}","{'precision': 0.7835051546391752, 'recall': 0.7835051546391752, 'f1': 0.7835051546391752, 'number': 97}","{'precision': 0.8604651162790697, 'recall': 0.9367088607594937, 'f1': 0.8969696969696971, 'number': 79}","{'precision': 0.9387755102040817, 'recall': 1.0, 'f1': 0.968421052631579, 'number': 92}","{'precision': 0.9120879120879121, 'recall': 0.9540229885057471, 'f1': 0.9325842696629213, 'number': 87}","{'precision': 0.975609756097561, 'recall': 0.96, 'f1': 0.9677419354838709, 'number': 125}","{'precision': 0.9658119658119658, 'recall': 1.0, 'f1': 0.9826086956521739, 'number': 113}","{'precision': 0.96, 'recall': 0.9473684210526315, 'f1': 0.9536423841059603, 'number': 228}","{'precision': 0.9959183673469387, 'recall': 0.976, 'f1': 0.9858585858585859, 'number': 250}",0.916493,0.919950,0.918218,0.

In [26]:
splits = ["loc", "reverse", "semantic", "heur_len", "heur_rare", "std", "advs"]

In [27]:
del df, records

In [ ]:
results = {}
trainer_all = {}
s = splits[0]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(cachaca_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: loc


/tmp/ipykernel_226035/4053031328.py:28: RuntimeWarning: invalid value encountered in divide
  topk[i] = (inter / uni).mean()
Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([35]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([35, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 2725/2725 [00:00<00:00, 13057.50 examples/s]
/tmp/ipykernel_226035/237942481.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Caracteristica Sensorial Aroma,Caracteristica Sensorial Consistência,Caracteristica Sensorial Cor,Caracteristica Sensorial Sabor,Classificacao Bebida,Equipamento Destilacao,Graduacao Alcoolica,Nome Bebida,Nome Local,Nome Organizacao,Nome Pessoa,Recipiente Armazenamento,Tempo,Tempo Armazenamento,Tipo Madeira,Volume,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.230700,0.144437,"{'precision': 0.55, 'recall': 0.55, 'f1': 0.55, 'number': 120}","{'precision': 0.8695652173913043, 'recall': 0.45454545454545453, 'f1': 0.5970149253731344, 'number': 44}","{'precision': 0.6875, 'recall': 0.6875, 'f1': 0.6875, 'number': 80}","{'precision': 0.5882352941176471, 'recall': 0.32, 'f1': 0.4145077720207254, 'number': 125}","{'precision': 0.8930817610062893, 'recall': 0.8352941176470589, 'f1': 0.8632218844984804, 'number': 170}","{'precision': 0.7708333333333334, 'recall': 0.7708333333333334, 'f1': 0.7708333333333333, 'number': 48}","{'precision': 0.8035714285714286, 'recall': 0.8490566037735849, 'f1': 0.8256880733944955, 'number': 53}","{'precision': 0.7294589178356713, 'recall': 0.7679324894514767, 'f1': 0.7482014388489209, 'number': 474}","{'precision': 0.8794326241134752, 'recall': 0.9323308270676691, 'f1': 0.9051094890510948, 'number': 532}","{'precision': 0.6178343949044586, 'recall': 0.6736111111111112, 'f1': 0.6445182724252491, 'number': 144}","{'precision': 0.839622641509434, 'recall': 0.9222797927461139, 'f1': 0.8790123456790123, 'number': 193}","{'precision': 0.9605263157894737, 'recall': 0.874251497005988, 'f1': 0.915360501567398, 'number': 167}","{'precision': 0.8756476683937824, 'recall': 0.8941798941798942, 'f1': 0.8848167539267016, 'number': 189}","{'precision': 0.8888888888888888, 'recall': 0.9624060150375939, 'f1': 0.924187725631769, 'number': 133}","{'precision': 0.9354838709677419, 'recall': 0.8907849829351536, 'f1': 0.9125874125874125, 'number': 293}","{'precision': 0.9672131147540983, 'recall': 0.9943820224719101, 'f1': 0.9806094182825486, 'number': 178}",0.824311,0.822630,0.823469,0.963873,0.963873,0.791554,0.962169
2,0.053900,0.145446,"{'precision': 0.45918367346938777, 'recall': 0.75, 'f1': 0.569620253164557, 'number': 120}","{'precision': 0.9333333333333333, 'recall': 0.6363636363636364, 'f1': 0.7567567567567568, 'number': 44}","{'precision': 0.6835443037974683, 'recall': 0.675, 'f1': 0.679245283018868, 'number': 80}","{'precision': 0.5581395348837209, 'recall': 0.384, 'f1': 0.4549763033175356, 'number': 125}","{'precision': 0.7559808612440191, 'recall': 0.9294117647058824, 'f1': 0.8337730870712401, 'number': 170}","{'precision': 0.7288135593220338, 'recall': 0.8958333333333334, 'f1': 0.8037383177570093, 'number': 48}","{'precision': 0.8571428571428571, 'recall': 0.9056603773584906, 'f1': 0.8807339449541285, 'number': 53}","{'precision': 0.717948717948718, 'recall': 0.7088607594936709, 'f1': 0.713375796178344, 'number': 474}","{'precision': 0.8363939899833055, 'recall': 0.9417293233082706, 'f1': 0.8859416445623342, 'number': 532}","{'precision': 0.5950920245398773, 'recall': 0.6736111111111112, 'f1': 0.6319218241042345, 'number': 144}","{'precision': 0.8894736842105263, 'recall': 0.8756476683937824, 'f1': 0.8825065274151437, 'number': 193}","{'precision': 0.9567901234567902, 'recall': 0.9281437125748503, 'f1': 0.9422492401215805, 'number': 167}","{'precision': 0.882051282051282, 'recall': 0.91005291005291, 'f1': 0.8958333333333333, 'number': 189}","{'precision': 0.9420289855072463, 'recall': 0.9774436090225563, 'f1': 0.9594095940959411, 'number': 133}","{'precision': 0.8544891640866873, 'recall': 0.9419795221843004, 'f1': 0.8961038961038961, 'number': 293}","{'precision': 0.9725274725274725, 'recall': 0.9943820224719101, 'f1': 0.9833333333333333, 'number': 178}",0.791707,0.843357,0.816716,0.964902,0.964902,0.804560,0.964406
3,0.035900,0.135790,"{'precision': 0.41397849462365593, 'recall': 0.6416666666666667, 'f1': 0.5032679738562091, 'num

/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


F1 Macro: 0.7701053003002256
F1 Micro: 0.9579240496671689
F1 Weighted: 0.9559846011861013
{'eval_loss': 0.20134669542312622, 'eval_CARACTERISTICA_SENSORIAL_AROMA': {'precision': 0.635593220338983, 'recall': 0.646551724137931, 'f1': 0.641025641025641, 'number': 464}, 'eval_CARACTERISTICA_SENSORIAL_CONSISTÊNCIA': {'precision': 0.8080808080808081, 'recall': 0.7339449541284404, 'f1': 0.7692307692307693, 'number': 109}, 'eval_CARACTERISTICA_SENSORIAL_COR': {'precision': 0.7692307692307693, 'recall': 0.7575757575757576, 'f1': 0.7633587786259541, 'number': 198}, 'eval_CARACTERISTICA_SENSORIAL_SABOR': {'precision': 0.5850746268656717, 'recall': 0.4336283185840708, 'f1': 0.49809402795425667, 'number': 452}, 'eval_CLASSIFICACAO_BEBIDA': {'precision': 0.7985347985347986, 'recall': 0.8790322580645161, 'f1': 0.8368522072936659, 'number': 248}, 'eval_EQUIPAMENTO_DESTILACAO': {'precision': 0.746268656716418, 'recall': 0.7692307692307693, 'f1': 0.7575757575757576, 'number': 65}, 'eval_GRADUACAO_ALCOOL

20

In [ ]:
import time

In [ ]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()


time.sleep(30)

In [ ]:
!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
results = {}
trainer_all = {}
s = splits[1]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(cachaca_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: reverse


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([35]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([35, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 2725/2725 [00:00<00:00, 11619.57 examples/s]
/tmp/ipykernel_226035/237942481.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Caracteristica Sensorial Aroma,Caracteristica Sensorial Consistência,Caracteristica Sensorial Cor,Caracteristica Sensorial Sabor,Classificacao Bebida,Equipamento Destilacao,Graduacao Alcoolica,Nome Bebida,Nome Local,Nome Organizacao,Nome Pessoa,Preco,Recipiente Armazenamento,Tempo,Tempo Armazenamento,Tipo Madeira,Volume,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.175700,1.081563,"{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 27}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 13}","{'precision': 0.7222222222222222, 'recall': 0.17105263157894737, 'f1': 0.2765957446808511, 'number': 76}","{'precision': 0.5961538461538461, 'recall': 0.25203252032520324, 'f1': 0.35428571428571426, 'number': 123}","{'precision': 0.9222222222222223, 'recall': 0.7614678899082569, 'f1': 0.834170854271357, 'number': 109}","{'precision': 0.6666666666666666, 'recall': 0.8571428571428571, 'f1': 0.75, 'number': 14}","{'precision': 0.6, 'recall': 0.7272727272727273, 'f1': 0.6575342465753425, 'number': 33}","{'precision': 0.6859205776173285, 'recall': 0.9134615384615384, 'f1': 0.7835051546391754, 'number': 208}","{'precision': 0.8670694864048338, 'recall': 0.875, 'f1': 0.8710166919575114, 'number': 328}","{'precision': 0.8679245283018868, 'recall': 0.5974025974025974, 'f1': 0.7076923076923077, 'number': 154}","{'precision': 0.8596491228070176, 'recall': 0.8032786885245902, 'f1': 0.8305084745762712, 'number': 61}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 559}","{'precision': 0.9354838709677419, 'recall': 0.8100558659217877, 'f1': 0.8682634730538922, 'number': 179}","{'precision': 0.9230769230769231, 'recall': 0.9302325581395349, 'f1': 0.9266409266409267, 'number': 129}","{'precision': 0.9047619047619048, 'recall': 0.9300699300699301, 'f1': 0.9172413793103449, 'number': 143}","{'precision': 0.8924418604651163, 'recall': 0.867231638418079, 'f1': 0.8796561604584526, 'number': 354}","{'precision': 0.08389261744966443, 'recall': 0.6578947368421053, 'f1': 0.1488095238095238, 'number': 152}",0.536354,0.595793,0.564513,0.839096,0.839096,0.609075,0.810705
2,0.040400,1.128797,"{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 27}","{'precision': 0.7142857142857143, 'recall': 0.38461538461538464, 'f1': 0.5, 'number': 13}","{'precision': 0.6, 'recall': 0.19736842105263158, 'f1': 0.297029702970297, 'number': 76}","{'precision': 0.6463414634146342, 'recall': 0.43089430894308944, 'f1': 0.5170731707317074, 'number': 123}","{'precision': 0.9222222222222223, 'recall': 0.7614678899082569, 'f1': 0.834170854271357, 'number': 109}","{'precision': 0.9230769230769231, 'recall': 0.8571428571428571, 'f1': 0.888888888888889, 'number': 14}","{'precision': 0.5945945945945946, 'recall': 0.6666666666666666, 'f1': 0.6285714285714286, 'number': 33}","{'precision': 0.7540322580645161, 'recall': 0.8990384615384616, 'f1': 0.8201754385964912, 'number': 208}","{'precision': 0.855072463768116, 'recall': 0.899390243902439, 'f1': 0.8766716196136702, 'number': 328}","{'precision': 0.7661290322580645, 'recall': 0.6168831168831169, 'f1': 0.6834532374100719, 'number': 154}","{'precision': 0.8529411764705882, 'recall': 0.9508196721311475, 'f1': 0.8992248062015503, 'number': 61}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 559}","{'precision': 0.9545454545454546, 'recall': 0.8212290502793296, 'f1': 0.8828828828828829, 'number': 179}","{'precision': 0.967741935483871, 'recall': 0.9302325581395349, 'f1': 0.9486166007905139, 'number': 129}","{'precision': 0.891156462585034, 'recall': 0.916083916083916, 'f1': 0.9034482758620689, 'number': 143}","{'precision': 0.9027027027027027, 'recall': 0.943502824858757, 'f1': 0.9226519337016574, 'number': 354}","{'precision': 0.08480268681780016, 'recall': 0.6644736842105263, 'f1': 0.15040953090096798, 'number': 152}",0.547918,0.622840,0.582982,0.844914,0.844914,0.645586,0.821992
3,0.022900,1.165828,"{'precision': 0.5454545454545454, 're

/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/me

/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


F1 Macro: 0.6200901480918215
F1 Micro: 0.9025458006186058
F1 Weighted: 0.8869578454241971
{'eval_loss': 0.6451143026351929, 'eval_CARACTERISTICA_SENSORIAL_AROMA': {'precision': 0.30612244897959184, 'recall': 0.10638297872340426, 'f1': 0.15789473684210528, 'number': 846}, 'eval_CARACTERISTICA_SENSORIAL_CONSISTÊNCIA': {'precision': 0.925531914893617, 'recall': 0.3551020408163265, 'f1': 0.5132743362831859, 'number': 245}, 'eval_CARACTERISTICA_SENSORIAL_COR': {'precision': 0.3096085409252669, 'recall': 0.20185614849187936, 'f1': 0.24438202247191013, 'number': 431}, 'eval_CARACTERISTICA_SENSORIAL_SABOR': {'precision': 0.38622754491017963, 'recall': 0.42644628099173554, 'f1': 0.40534171249018064, 'number': 605}, 'eval_CLASSIFICACAO_BEBIDA': {'precision': 0.8920863309352518, 'recall': 0.5188284518828452, 'f1': 0.6560846560846562, 'number': 478}, 'eval_EQUIPAMENTO_DESTILACAO': {'precision': 0.8156028368794326, 'recall': 0.5111111111111111, 'f1': 0.628415300546448, 'number': 225}, 'eval_GRADUAC

0

In [ ]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()


time.sleep(30)


!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
results = {}
trainer_all = {}
s = splits[2]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(cachaca_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: semantic


100%|██████████| 13628/13628 [00:00<00:00, 2178352.70it/s]
Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([35]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([35, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 2780/2780 [00:00<00:00, 34584.40 examples/s]
/tmp/ipykernel_226035/237942481.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Caracteristica Sensorial Consistência,Caracteristica Sensorial Cor,Caracteristica Sensorial Sabor,Classificacao Bebida,Graduacao Alcoolica,Nome Bebida,Nome Local,Nome Organizacao,Nome Pessoa,Preco,Recipiente Armazenamento,Tempo,Tempo Armazenamento,Tipo Madeira,Volume,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.245200,0.649442,"{'precision': 1.0, 'recall': 0.375, 'f1': 0.5454545454545454, 'number': 8}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 5}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 2}","{'precision': 0.9333333333333333, 'recall': 0.5, 'f1': 0.6511627906976745, 'number': 56}","{'precision': 0.9382022471910112, 'recall': 0.9842829076620825, 'f1': 0.9606903163950143, 'number': 509}","{'precision': 0.8, 'recall': 0.8, 'f1': 0.8000000000000002, 'number': 20}","{'precision': 0.9830508474576272, 'recall': 0.9863945578231292, 'f1': 0.9847198641765705, 'number': 294}","{'precision': 1.0, 'recall': 0.8888888888888888, 'f1': 0.9411764705882353, 'number': 9}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 3}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 172}","{'precision': 0.7272727272727273, 'recall': 1.0, 'f1': 0.8421052631578948, 'number': 8}","{'precision': 0.9710144927536232, 'recall': 1.0, 'f1': 0.9852941176470589, 'number': 67}","{'precision': 0.9166666666666666, 'recall': 1.0, 'f1': 0.9565217391304348, 'number': 11}","{'precision': 0.8562874251497006, 'recall': 0.9930555555555556, 'f1': 0.9196141479099679, 'number': 144}","{'precision': 0.96875, 'recall': 0.6595744680851063, 'f1': 0.7848101265822784, 'number': 47}",0.937028,0.823616,0.876669,0.902797,0.902797,0.825088,0.863627
2,0.062600,0.849967,"{'precision': 1.0, 'recall': 0.375, 'f1': 0.5454545454545454, 'number': 8}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 5}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 2}","{'precision': 1.0, 'recall': 0.5357142857142857, 'f1': 0.6976744186046512, 'number': 56}","{'precision': 0.9770354906054279, 'recall': 0.9194499017681729, 'f1': 0.9473684210526315, 'number': 509}","{'precision': 0.9047619047619048, 'recall': 0.95, 'f1': 0.9268292682926829, 'number': 20}","{'precision': 0.9831081081081081, 'recall': 0.9897959183673469, 'f1': 0.9864406779661017, 'number': 294}","{'precision': 1.0, 'recall': 0.8888888888888888, 'f1': 0.9411764705882353, 'number': 9}","{'precision': 1.0, 'recall': 0.6666666666666666, 'f1': 0.8, 'number': 3}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 172}","{'precision': 0.7272727272727273, 'recall': 1.0, 'f1': 0.8421052631578948, 'number': 8}","{'precision': 0.9852941176470589, 'recall': 1.0, 'f1': 0.9925925925925926, 'number': 67}","{'precision': 0.8333333333333334, 'recall': 0.9090909090909091, 'f1': 0.8695652173913043, 'number': 11}","{'precision': 0.8614457831325302, 'recall': 0.9930555555555556, 'f1': 0.9225806451612903, 'number': 144}","{'precision': 1.0, 'recall': 0.6595744680851063, 'f1': 0.7948717948717948, 'number': 47}",0.958554,0.802214,0.873443,0.899445,0.899445,0.786325,0.860513
3,0.036000,0.753524,"{'precision': 1.0, 'recall': 0.5, 'f1': 0.6666666666666666, 'number': 8}","{'precision': 0.22727272727272727, 'recall': 1.0, 'f1': 0.37037037037037035, 'number': 5}","{'precision': 0.6666666666666666, 'recall': 1.0, 'f1': 0.8, 'number': 2}","{'precision': 1.0, 'recall': 0.6964285714285714, 'f1': 0.8210526315789474, 'number': 56}","{'precision': 0.9353049907578558, 'recall': 0.9941060903732809, 'f1': 0.9638095238095237, 'number': 509}","{'precision': 0.9047619047619048, 'recall': 0.95, 'f1': 0.9268292682926829, 'number': 20}","{'precision': 0.9765100671140939, 'recall': 0.9897959183673469, 'f1': 0.9831081081081081, 'number': 294}","{'precision': 1.0, 'recall': 0.8888888888888888, 'f1': 0.9411764705882353, 'number': 9}","{'precision': 1.0, 'recall': 0.6666666666666666, 'f1': 0.8, 'number': 3}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0,

/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/me

/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


F1 Macro: 0.6423460961331603
F1 Micro: 0.772513311212285
F1 Weighted: 0.6858358576906439
{'eval_loss': 2.1143741607666016, 'eval_CARACTERISTICA_SENSORIAL_AROMA': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 0}, 'eval_CARACTERISTICA_SENSORIAL_CONSISTÊNCIA': {'precision': 1.0, 'recall': 0.8, 'f1': 0.888888888888889, 'number': 50}, 'eval_CARACTERISTICA_SENSORIAL_SABOR': {'precision': 0.75, 'recall': 0.8709677419354839, 'f1': 0.8059701492537312, 'number': 31}, 'eval_CLASSIFICACAO_BEBIDA': {'precision': 0.6428571428571429, 'recall': 0.5294117647058824, 'f1': 0.5806451612903226, 'number': 17}, 'eval_GRADUACAO_ALCOOLICA': {'precision': 0.7995824634655533, 'recall': 0.9896640826873385, 'f1': 0.884526558891455, 'number': 387}, 'eval_NOME_BEBIDA': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 12}, 'eval_NOME_LOCAL': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 244}, 'eval_PRECO': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 710}, 'eval_RECIPIENTE_ARMAZENAME

0

In [ ]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()


time.sleep(30)


!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
results = {}
trainer_all = {}
s = splits[3]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(cachaca_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: heur_len


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([35]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([35, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 2725/2725 [00:00<00:00, 15523.05 examples/s]
/tmp/ipykernel_226035/237942481.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Caracteristica Sensorial Aroma,Caracteristica Sensorial Consistência,Caracteristica Sensorial Cor,Caracteristica Sensorial Sabor,Classificacao Bebida,Equipamento Destilacao,Graduacao Alcoolica,Nome Bebida,Nome Local,Nome Organizacao,Nome Pessoa,Preco,Recipiente Armazenamento,Tempo,Tempo Armazenamento,Tipo Madeira,Volume,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.259600,0.091624,"{'precision': 0.6869565217391305, 'recall': 0.8977272727272727, 'f1': 0.7783251231527094, 'number': 88}","{'precision': 1.0, 'recall': 0.6774193548387096, 'f1': 0.8076923076923077, 'number': 31}","{'precision': 0.660377358490566, 'recall': 0.660377358490566, 'f1': 0.660377358490566, 'number': 53}","{'precision': 0.4418604651162791, 'recall': 0.5352112676056338, 'f1': 0.4840764331210191, 'number': 71}","{'precision': 0.8553459119496856, 'recall': 0.9379310344827586, 'f1': 0.8947368421052632, 'number': 145}","{'precision': 0.9259259259259259, 'recall': 1.0, 'f1': 0.9615384615384615, 'number': 25}","{'precision': 0.9444444444444444, 'recall': 0.9834710743801653, 'f1': 0.9635627530364373, 'number': 121}","{'precision': 0.8299120234604106, 'recall': 0.881619937694704, 'f1': 0.8549848942598188, 'number': 321}","{'precision': 0.9546436285097192, 'recall': 0.9505376344086022, 'f1': 0.9525862068965517, 'number': 465}","{'precision': 0.794392523364486, 'recall': 0.7727272727272727, 'f1': 0.7834101382488479, 'number': 110}","{'precision': 0.8446601941747572, 'recall': 0.87, 'f1': 0.8571428571428571, 'number': 100}","{'precision': 0.9529411764705882, 'recall': 1.0, 'f1': 0.9759036144578312, 'number': 81}","{'precision': 0.9489795918367347, 'recall': 0.96875, 'f1': 0.9587628865979382, 'number': 96}","{'precision': 0.9523809523809523, 'recall': 0.9411764705882353, 'f1': 0.9467455621301775, 'number': 170}","{'precision': 0.9401709401709402, 'recall': 1.0, 'f1': 0.9691629955947136, 'number': 110}","{'precision': 0.9224489795918367, 'recall': 0.9535864978902954, 'f1': 0.9377593360995851, 'number': 237}","{'precision': 0.9871244635193133, 'recall': 0.9829059829059829, 'f1': 0.9850107066381155, 'number': 234}",0.883392,0.915378,0.899101,0.975894,0.975894,0.855253,0.975708
2,0.071000,0.075174,"{'precision': 0.7717391304347826, 'recall': 0.8068181818181818, 'f1': 0.7888888888888889, 'number': 88}","{'precision': 0.92, 'recall': 0.7419354838709677, 'f1': 0.8214285714285714, 'number': 31}","{'precision': 0.803921568627451, 'recall': 0.7735849056603774, 'f1': 0.7884615384615384, 'number': 53}","{'precision': 0.7066666666666667, 'recall': 0.7464788732394366, 'f1': 0.7260273972602739, 'number': 71}","{'precision': 0.9144736842105263, 'recall': 0.9586206896551724, 'f1': 0.936026936026936, 'number': 145}","{'precision': 0.8064516129032258, 'recall': 1.0, 'f1': 0.8928571428571428, 'number': 25}","{'precision': 0.9836065573770492, 'recall': 0.9917355371900827, 'f1': 0.9876543209876544, 'number': 121}","{'precision': 0.8776119402985074, 'recall': 0.9158878504672897, 'f1': 0.8963414634146342, 'number': 321}","{'precision': 0.9553191489361702, 'recall': 0.9655913978494624, 'f1': 0.960427807486631, 'number': 465}","{'precision': 0.8053097345132744, 'recall': 0.8272727272727273, 'f1': 0.8161434977578476, 'number': 110}","{'precision': 0.8865979381443299, 'recall': 0.86, 'f1': 0.8730964467005077, 'number': 100}","{'precision': 0.9642857142857143, 'recall': 1.0, 'f1': 0.9818181818181818, 'number': 81}","{'precision': 0.9108910891089109, 'recall': 0.9583333333333334, 'f1': 0.934010152284264, 'number': 96}","{'precision': 0.9364161849710982, 'recall': 0.9529411764705882, 'f1': 0.944606413994169, 'number': 170}","{'precision': 0.9565217391304348, 'recall': 1.0, 'f1': 0.9777777777777777, 'number': 110}","{'precision': 0.9453781512605042, 'recall': 0.9493670886075949, 'f1': 0.9473684210526315, 'number': 237}","{'precision': 0.9829059829059829, 'recall': 0.9829059829059829, 'f1': 0.9829059829059829, 'number'

F1 Macro: 0.9257355012615386
F1 Micro: 0.9842191780821917
F1 Weighted: 0.9842097614072365
{'eval_loss': 0.06349194794893265, 'eval_CARACTERISTICA_SENSORIAL_AROMA': {'precision': 0.7594339622641509, 'recall': 0.8214285714285714, 'f1': 0.7892156862745098, 'number': 196}, 'eval_CARACTERISTICA_SENSORIAL_CONSISTÊNCIA': {'precision': 0.9019607843137255, 'recall': 0.8846153846153846, 'f1': 0.8932038834951457, 'number': 52}, 'eval_CARACTERISTICA_SENSORIAL_COR': {'precision': 0.9327731092436975, 'recall': 0.9327731092436975, 'f1': 0.9327731092436976, 'number': 119}, 'eval_CARACTERISTICA_SENSORIAL_SABOR': {'precision': 0.8050314465408805, 'recall': 0.7314285714285714, 'f1': 0.7664670658682634, 'number': 175}, 'eval_CLASSIFICACAO_BEBIDA': {'precision': 0.9444444444444444, 'recall': 0.9732824427480916, 'f1': 0.9586466165413535, 'number': 262}, 'eval_EQUIPAMENTO_DESTILACAO': {'precision': 0.9242424242424242, 'recall': 0.9384615384615385, 'f1': 0.9312977099236641, 'number': 65}, 'eval_GRADUACAO_ALCO

0

In [ ]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()


time.sleep(30)


!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
results = {}
trainer_all = {}
s = splits[4]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(cachaca_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: heur_rare


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([35]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([35, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 1729/1729 [00:00<00:00, 10122.92 examples/s]
/tmp/ipykernel_226035/237942481.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Caracteristica Sensorial Aroma,Caracteristica Sensorial Consistência,Caracteristica Sensorial Cor,Caracteristica Sensorial Sabor,Classificacao Bebida,Equipamento Destilacao,Graduacao Alcoolica,Nome Bebida,Nome Local,Nome Organizacao,Nome Pessoa,Preco,Recipiente Armazenamento,Tempo,Tempo Armazenamento,Tipo Madeira,Volume,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.248800,0.063430,"{'precision': 0.7878787878787878, 'recall': 0.6419753086419753, 'f1': 0.7074829931972788, 'number': 81}","{'precision': 0.90625, 'recall': 0.8787878787878788, 'f1': 0.8923076923076922, 'number': 33}","{'precision': 0.9622641509433962, 'recall': 0.8947368421052632, 'f1': 0.9272727272727272, 'number': 57}","{'precision': 0.5904761904761905, 'recall': 0.8266666666666667, 'f1': 0.6888888888888889, 'number': 75}","{'precision': 0.9481481481481482, 'recall': 0.8951048951048951, 'f1': 0.920863309352518, 'number': 143}","{'precision': 0.875, 'recall': 0.9333333333333333, 'f1': 0.9032258064516129, 'number': 30}","{'precision': 0.9927007299270073, 'recall': 1.0, 'f1': 0.9963369963369962, 'number': 136}","{'precision': 0.8367952522255193, 'recall': 0.912621359223301, 'f1': 0.8730650154798761, 'number': 309}","{'precision': 0.9665071770334929, 'recall': 0.9711538461538461, 'f1': 0.9688249400479617, 'number': 416}","{'precision': 0.9411764705882353, 'recall': 0.8727272727272727, 'f1': 0.9056603773584905, 'number': 110}","{'precision': 0.9710144927536232, 'recall': 0.9710144927536232, 'f1': 0.9710144927536232, 'number': 69}","{'precision': 0.9772727272727273, 'recall': 1.0, 'f1': 0.9885057471264368, 'number': 86}","{'precision': 0.9186046511627907, 'recall': 0.9753086419753086, 'f1': 0.9461077844311376, 'number': 81}","{'precision': 0.9739130434782609, 'recall': 0.9911504424778761, 'f1': 0.9824561403508772, 'number': 113}","{'precision': 0.9734513274336283, 'recall': 1.0, 'f1': 0.9865470852017937, 'number': 110}","{'precision': 0.967032967032967, 'recall': 0.9814126394052045, 'f1': 0.974169741697417, 'number': 269}","{'precision': 1.0, 'recall': 0.9930795847750865, 'f1': 0.9965277777777778, 'number': 289}",0.928513,0.944329,0.936354,0.984135,0.984135,0.902096,0.984133
2,0.064600,0.046566,"{'precision': 0.881578947368421, 'recall': 0.8271604938271605, 'f1': 0.8535031847133758, 'number': 81}","{'precision': 0.9117647058823529, 'recall': 0.9393939393939394, 'f1': 0.9253731343283583, 'number': 33}","{'precision': 0.9411764705882353, 'recall': 0.8421052631578947, 'f1': 0.8888888888888888, 'number': 57}","{'precision': 0.7837837837837838, 'recall': 0.7733333333333333, 'f1': 0.7785234899328859, 'number': 75}","{'precision': 0.9645390070921985, 'recall': 0.951048951048951, 'f1': 0.9577464788732395, 'number': 143}","{'precision': 0.9629629629629629, 'recall': 0.8666666666666667, 'f1': 0.912280701754386, 'number': 30}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 136}","{'precision': 0.878125, 'recall': 0.9093851132686084, 'f1': 0.8934817170111288, 'number': 309}","{'precision': 0.9758454106280193, 'recall': 0.9711538461538461, 'f1': 0.9734939759036145, 'number': 416}","{'precision': 0.8981481481481481, 'recall': 0.8818181818181818, 'f1': 0.8899082568807339, 'number': 110}","{'precision': 0.9710144927536232, 'recall': 0.9710144927536232, 'f1': 0.9710144927536232, 'number': 69}","{'precision': 0.9772727272727273, 'recall': 1.0, 'f1': 0.9885057471264368, 'number': 86}","{'precision': 0.9404761904761905, 'recall': 0.9753086419753086, 'f1': 0.9575757575757574, 'number': 81}","{'precision': 0.9824561403508771, 'recall': 0.9911504424778761, 'f1': 0.9867841409691629, 'number': 113}","{'precision': 0.972972972972973, 'recall': 0.9818181818181818, 'f1': 0.9773755656108598, 'number': 110}","{'precision': 0.9743589743589743, 'recall': 0.9888475836431226, 'f1': 0.9815498154981549, 'number': 269}","{'precision': 1.0, 'recall': 0.9896193771626297, 'f1': 0.994782608695652, 'number': 289}",0.95

F1 Macro: 0.8456001143690215
F1 Micro: 0.9661714050691481
F1 Weighted: 0.966100993723563
{'eval_loss': 0.14895674586296082, 'eval_CARACTERISTICA_SENSORIAL_AROMA': {'precision': 0.7327586206896551, 'recall': 0.7172995780590717, 'f1': 0.7249466950959489, 'number': 237}, 'eval_CARACTERISTICA_SENSORIAL_CONSISTÊNCIA': {'precision': 0.8166666666666667, 'recall': 0.765625, 'f1': 0.7903225806451613, 'number': 64}, 'eval_CARACTERISTICA_SENSORIAL_COR': {'precision': 0.7047619047619048, 'recall': 0.7956989247311828, 'f1': 0.7474747474747475, 'number': 93}, 'eval_CARACTERISTICA_SENSORIAL_SABOR': {'precision': 0.5941422594142259, 'recall': 0.6255506607929515, 'f1': 0.609442060085837, 'number': 227}, 'eval_CLASSIFICACAO_BEBIDA': {'precision': 0.8229166666666666, 'recall': 0.8926553672316384, 'f1': 0.8563685636856369, 'number': 177}, 'eval_EQUIPAMENTO_DESTILACAO': {'precision': 0.8103448275862069, 'recall': 0.8392857142857143, 'f1': 0.8245614035087718, 'number': 56}, 'eval_GRADUACAO_ALCOOLICA': {'pre

0

In [ ]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()


time.sleep(30)


!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [28]:
results = {}
trainer_all = {}
s = splits[5]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(cachaca_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: std


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([35]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([35, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 2727/2727 [00:00<00:00, 16376.77 examples/s]
C:\Users\user\AppData\Local\Temp\ipykernel_15624\2810120630.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Caracteristica Sensorial Aroma,Caracteristica Sensorial Consistência,Caracteristica Sensorial Cor,Caracteristica Sensorial Sabor,Classificacao Bebida,Equipamento Destilacao,Graduacao Alcoolica,Nome Bebida,Nome Local,Nome Organizacao,Nome Pessoa,Preco,Recipiente Armazenamento,Tempo,Tempo Armazenamento,Tipo Madeira,Volume,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.260300,0.096307,"{'precision': 0.7610619469026548, 'recall': 0.7678571428571429, 'f1': 0.7644444444444445, 'number': 112}","{'precision': 0.8787878787878788, 'recall': 0.7631578947368421, 'f1': 0.8169014084507042, 'number': 38}","{'precision': 0.8545454545454545, 'recall': 0.8245614035087719, 'f1': 0.8392857142857144, 'number': 57}","{'precision': 0.6075949367088608, 'recall': 0.5517241379310345, 'f1': 0.5783132530120483, 'number': 87}","{'precision': 0.8870967741935484, 'recall': 0.8661417322834646, 'f1': 0.8764940239043826, 'number': 127}","{'precision': 0.8214285714285714, 'recall': 0.8846153846153846, 'f1': 0.8518518518518519, 'number': 26}","{'precision': 0.9545454545454546, 'recall': 0.9722222222222222, 'f1': 0.963302752293578, 'number': 108}","{'precision': 0.8626865671641791, 'recall': 0.85, 'f1': 0.8562962962962963, 'number': 340}","{'precision': 0.9395348837209302, 'recall': 0.9619047619047619, 'f1': 0.9505882352941177, 'number': 420}","{'precision': 0.7450980392156863, 'recall': 0.7835051546391752, 'f1': 0.763819095477387, 'number': 97}","{'precision': 0.7708333333333334, 'recall': 0.9367088607594937, 'f1': 0.8457142857142858, 'number': 79}","{'precision': 0.92, 'recall': 1.0, 'f1': 0.9583333333333334, 'number': 92}","{'precision': 0.9032258064516129, 'recall': 0.9655172413793104, 'f1': 0.9333333333333333, 'number': 87}","{'precision': 0.905982905982906, 'recall': 0.848, 'f1': 0.8760330578512396, 'number': 125}","{'precision': 0.835820895522388, 'recall': 0.9911504424778761, 'f1': 0.9068825910931174, 'number': 113}","{'precision': 0.9243697478991597, 'recall': 0.9649122807017544, 'f1': 0.944206008583691, 'number': 228}","{'precision': 0.9718875502008032, 'recall': 0.968, 'f1': 0.969939879759519, 'number': 250}",0.881363,0.899832,0.890502,0.974066,0.974066,0.853830,0.973663
2,0.073300,0.080688,"{'precision': 0.6643356643356644, 'recall': 0.8482142857142857, 'f1': 0.7450980392156863, 'number': 112}","{'precision': 0.8205128205128205, 'recall': 0.8421052631578947, 'f1': 0.8311688311688312, 'number': 38}","{'precision': 0.8703703703703703, 'recall': 0.8245614035087719, 'f1': 0.8468468468468469, 'number': 57}","{'precision': 0.7592592592592593, 'recall': 0.47126436781609193, 'f1': 0.5815602836879432, 'number': 87}","{'precision': 0.8705035971223022, 'recall': 0.952755905511811, 'f1': 0.9097744360902256, 'number': 127}","{'precision': 0.96, 'recall': 0.9230769230769231, 'f1': 0.9411764705882353, 'number': 26}","{'precision': 0.9814814814814815, 'recall': 0.9814814814814815, 'f1': 0.9814814814814815, 'number': 108}","{'precision': 0.8982558139534884, 'recall': 0.9088235294117647, 'f1': 0.9035087719298247, 'number': 340}","{'precision': 0.9641148325358851, 'recall': 0.9595238095238096, 'f1': 0.9618138424821002, 'number': 420}","{'precision': 0.7222222222222222, 'recall': 0.8041237113402062, 'f1': 0.7609756097560977, 'number': 97}","{'precision': 0.8352941176470589, 'recall': 0.8987341772151899, 'f1': 0.8658536585365854, 'number': 79}","{'precision': 0.9387755102040817, 'recall': 1.0, 'f1': 0.968421052631579, 'number': 92}","{'precision': 0.9333333333333333, 'recall': 0.9655172413793104, 'f1': 0.9491525423728815, 'number': 87}","{'precision': 0.9758064516129032, 'recall': 0.968, 'f1': 0.9718875502008033, 'number': 125}","{'precision': 0.9741379310344828, 'recall': 1.0, 'f1': 0.9868995633187774, 'number': 113}","{'precision': 0.9644444444444444, 'recall': 0.9517543859649122, 'f1': 0.9580573951434879, 'number': 228}","{'precision': 0.9959183673469387, 'recall': 0.976, 'f1': 0.985858

F1 Macro: 0.9351310626522764
F1 Micro: 0.9863269111249223
F1 Weighted: 0.9864214286145429
{'eval_loss': 0.060617782175540924, 'eval_CARACTERISTICA_SENSORIAL_AROMA': {'precision': 0.7891891891891892, 'recall': 0.8488372093023255, 'f1': 0.8179271708683473, 'number': 172}, 'eval_CARACTERISTICA_SENSORIAL_CONSISTÊNCIA': {'precision': 0.9285714285714286, 'recall': 0.8666666666666667, 'f1': 0.896551724137931, 'number': 45}, 'eval_CARACTERISTICA_SENSORIAL_COR': {'precision': 0.8672566371681416, 'recall': 0.8521739130434782, 'f1': 0.8596491228070176, 'number': 115}, 'eval_CARACTERISTICA_SENSORIAL_SABOR': {'precision': 0.7409638554216867, 'recall': 0.7735849056603774, 'f1': 0.756923076923077, 'number': 159}, 'eval_CLASSIFICACAO_BEBIDA': {'precision': 0.9259259259259259, 'recall': 0.9821428571428571, 'f1': 0.9532062391681109, 'number': 280}, 'eval_EQUIPAMENTO_DESTILACAO': {'precision': 0.921875, 'recall': 0.921875, 'f1': 0.921875, 'number': 64}, 'eval_GRADUACAO_ALCOOLICA': {'precision': 0.9823008

40

In [31]:
import time

In [ ]:
#
# del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()


time.sleep(30)


!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

Acesso negado - .
Arquivo n�o encontrado  - -NAME
Arquivo n�o encontrado  - -EXEC
Arquivo n�o encontrado  - RM
Arquivo n�o encontrado  - -RF
Arquivo n�o encontrado  - {}
Arquivo n�o encontrado  - +


In [33]:
results = {}
trainer_all = {}
s = splits[6]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(cachaca_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: advs
Selecionando 2725 sentenças para teste…
  0 selecionadas…
  27 selecionadas…
  52 selecionadas…
  77 selecionadas…
  102 selecionadas…
  127 selecionadas…
  152 selecionadas…
  177 selecionadas…
  203 selecionadas…
  229 selecionadas…
  256 selecionadas…
  284 selecionadas…
  309 selecionadas…
  336 selecionadas…
  359 selecionadas…
  384 selecionadas…
  408 selecionadas…
  430 selecionadas…
  451 selecionadas…
  475 selecionadas…
  498 selecionadas…
  525 selecionadas…
  550 selecionadas…
  571 selecionadas…
  591 selecionadas…
  609 selecionadas…
  631 selecionadas…
  648 selecionadas…
  667 selecionadas…
  693 selecionadas…
  708 selecionadas…
  730 selecionadas…
  749 selecionadas…
  763 selecionadas…
  772 selecionadas…
  785 selecionadas…
  799 selecionadas…
  823 selecionadas…
  845 selecionadas…
  862 selecionadas…
  884 selecionadas…
  907 selecionadas…
  927 selecionadas…
  948 selecionadas…
  971 selecionadas…
  992 selecionadas…
  1014 selecionadas

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([35]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([35, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 2725/2725 [00:00<00:00, 5346.41 examples/s]
C:\Users\user\AppData\Local\Temp\ipykernel_15624\2810120630.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Caracteristica Sensorial Aroma,Caracteristica Sensorial Consistência,Caracteristica Sensorial Cor,Caracteristica Sensorial Sabor,Classificacao Bebida,Equipamento Destilacao,Graduacao Alcoolica,Nome Bebida,Nome Local,Nome Organizacao,Nome Pessoa,Preco,Recipiente Armazenamento,Tempo,Tempo Armazenamento,Tipo Madeira,Volume,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.255300,0.090937,"{'precision': 0.6923076923076923, 'recall': 0.3698630136986301, 'f1': 0.48214285714285715, 'number': 73}","{'precision': 0.8181818181818182, 'recall': 0.6428571428571429, 'f1': 0.7200000000000001, 'number': 28}","{'precision': 0.94, 'recall': 0.8392857142857143, 'f1': 0.8867924528301886, 'number': 56}","{'precision': 0.5033557046979866, 'recall': 0.78125, 'f1': 0.6122448979591837, 'number': 96}","{'precision': 0.9508196721311475, 'recall': 0.8854961832061069, 'f1': 0.9169960474308301, 'number': 131}","{'precision': 0.8636363636363636, 'recall': 0.7307692307692307, 'f1': 0.7916666666666666, 'number': 26}","{'precision': 0.952, 'recall': 0.9834710743801653, 'f1': 0.967479674796748, 'number': 121}","{'precision': 0.8440677966101695, 'recall': 0.8556701030927835, 'f1': 0.8498293515358363, 'number': 291}","{'precision': 0.9502262443438914, 'recall': 0.958904109589041, 'f1': 0.9545454545454545, 'number': 438}","{'precision': 0.7916666666666666, 'recall': 0.8260869565217391, 'f1': 0.8085106382978724, 'number': 92}","{'precision': 0.8735632183908046, 'recall': 0.9620253164556962, 'f1': 0.9156626506024097, 'number': 79}","{'precision': 0.8877551020408163, 'recall': 1.0, 'f1': 0.9405405405405405, 'number': 87}","{'precision': 0.9493670886075949, 'recall': 0.8928571428571429, 'f1': 0.920245398773006, 'number': 84}","{'precision': 0.910958904109589, 'recall': 0.9236111111111112, 'f1': 0.9172413793103449, 'number': 144}","{'precision': 0.9819819819819819, 'recall': 1.0, 'f1': 0.9909090909090909, 'number': 109}","{'precision': 0.9317269076305221, 'recall': 0.9586776859504132, 'f1': 0.9450101832993891, 'number': 242}","{'precision': 0.9961389961389961, 'recall': 0.9555555555555556, 'f1': 0.9754253308128544, 'number': 270}",0.893350,0.902408,0.897856,0.974552,0.974552,0.858537,0.973942
2,0.067200,0.083756,"{'precision': 0.6506024096385542, 'recall': 0.7397260273972602, 'f1': 0.6923076923076923, 'number': 73}","{'precision': 0.8, 'recall': 0.8571428571428571, 'f1': 0.8275862068965518, 'number': 28}","{'precision': 0.8928571428571429, 'recall': 0.8928571428571429, 'f1': 0.8928571428571429, 'number': 56}","{'precision': 0.8051948051948052, 'recall': 0.6458333333333334, 'f1': 0.7167630057803468, 'number': 96}","{'precision': 0.9761904761904762, 'recall': 0.9389312977099237, 'f1': 0.9571984435797665, 'number': 131}","{'precision': 0.896551724137931, 'recall': 1.0, 'f1': 0.9454545454545454, 'number': 26}","{'precision': 0.9523809523809523, 'recall': 0.9917355371900827, 'f1': 0.97165991902834, 'number': 121}","{'precision': 0.9027777777777778, 'recall': 0.8934707903780069, 'f1': 0.8981001727115717, 'number': 291}","{'precision': 0.9428571428571428, 'recall': 0.9794520547945206, 'f1': 0.9608062709966406, 'number': 438}","{'precision': 0.7731958762886598, 'recall': 0.8152173913043478, 'f1': 0.7936507936507937, 'number': 92}","{'precision': 0.872093023255814, 'recall': 0.9493670886075949, 'f1': 0.9090909090909091, 'number': 79}","{'precision': 0.8877551020408163, 'recall': 1.0, 'f1': 0.9405405405405405, 'number': 87}","{'precision': 0.9529411764705882, 'recall': 0.9642857142857143, 'f1': 0.9585798816568047, 'number': 84}","{'precision': 0.9513888888888888, 'recall': 0.9513888888888888, 'f1': 0.9513888888888888, 'number': 144}","{'precision': 0.9732142857142857, 'recall': 1.0, 'f1': 0.9864253393665159, 'number': 109}","{'precision': 0.9634146341463414, 'recall': 0.9793388429752066, 'f1': 0.971311475409836, 'number': 242}","{'precision': 0.9885496183206107, 'recall': 0.9592592592592593, 'f1': 

F1 Macro: 0.8715991846341229
F1 Micro: 0.9764015037181352
F1 Weighted: 0.9763144019659569
{'eval_loss': 0.10785236954689026, 'eval_CARACTERISTICA_SENSORIAL_AROMA': {'precision': 0.7768924302788844, 'recall': 0.819327731092437, 'f1': 0.7975460122699387, 'number': 238}, 'eval_CARACTERISTICA_SENSORIAL_CONSISTÊNCIA': {'precision': 0.8493150684931506, 'recall': 0.8732394366197183, 'f1': 0.861111111111111, 'number': 71}, 'eval_CARACTERISTICA_SENSORIAL_COR': {'precision': 0.9067796610169492, 'recall': 0.8699186991869918, 'f1': 0.8879668049792531, 'number': 123}, 'eval_CARACTERISTICA_SENSORIAL_SABOR': {'precision': 0.7564766839378239, 'recall': 0.7684210526315789, 'f1': 0.7624020887728459, 'number': 190}, 'eval_CLASSIFICACAO_BEBIDA': {'precision': 0.8864468864468864, 'recall': 0.968, 'f1': 0.9254302103250478, 'number': 250}, 'eval_EQUIPAMENTO_DESTILACAO': {'precision': 0.8082191780821918, 'recall': 0.921875, 'f1': 0.8613138686131386, 'number': 64}, 'eval_GRADUACAO_ALCOOLICA': {'precision': 0.9

40